#Project FRM-GIS

## 1. Initialize Google Earth Engine

Load the Google Cloud Project ID from Colab Secrets, authenticate
with Google Earth Engine, and verify that the connection works.

In [ ]:
%pip install -q geemap

In [ ]:
!pip install earthengine-api geemap

In [ ]:
import ee
from google.colab import userdata

PROJECT_ID = userdata.get("PROJECT_ID")

if not PROJECT_ID:
    raise ValueError(
        "PROJECT_ID not found. Check Colab Secrets and notebook access."
    )

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print(ee.String("Earth Engine connection successful").getInfo())


In [ ]:
# Verify that the SRTM 30 m elevation dataset is accessible
srtm = ee.Image("USGS/SRTMGL1_003")

print("Dataset:", "SRTM 30 m DEM")
print("Available bands:", srtm.bandNames().getInfo())
print("Dataset access verified.")

##2.1 Inspect candidate watershed boundaries

Use HydroBASINS Level 7 to identify candidate catchments around
the Pamba River Basin. Inspect the results before selecting an AOI.

In [ ]:
import geemap

# Approximate search window around the Pamba Basin.
# This is only for finding candidate catchments, NOT the final AOI.
search_area = ee.Geometry.Rectangle([76.2, 8.5, 77.4, 10.2])

# Load candidate watershed polygons
basins = ee.FeatureCollection(
    "WWF/HydroSHEDS/v1/Basins/hybas_7"
).filterBounds(search_area)

print("Candidate basin polygons:", basins.size().getInfo())

# Inspect available attributes
print(
    "First candidate attributes:",
    basins.first().toDictionary().getInfo()
)

In [ ]:
Map = geemap.Map(
    basemap="Esri.WorldTopoMap"
)

Map.centerObject(search_area, 8)

Map.addLayer(
    search_area,
    {"color": "yellow"},
    "Search window"
)

Map.addLayer(
    basins.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "HydroBASINS candidates"
)

Map

## 2.2 Inspect HydroBASINS Candidate Attributes

Check the number of candidate features and their available
properties before selecting the Pamba River Basin boundary.

In [ ]:
# Count candidate basin features
print("Number of basin candidates:", basins.size().getInfo())

# Inspect available attributes
first_basin = ee.Feature(basins.first())

print("\nAvailable properties:")
print(first_basin.propertyNames().getInfo())

# Display a small sample of attributes without geometries
sample = (
    basins
    .limit(5)
    .select(first_basin.propertyNames(), None, False)
)

print("\nFirst five candidate records:")
print(sample.getInfo()["features"])

## 2.3 Compare HydroBASINS Candidates

Create a compact attribute table for all candidate polygons
to support evidence-based selection of the Pamba River Basin.

In [ ]:
import pandas as pd

fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "MAIN_BAS",
    "NEXT_DOWN",
    "DIST_MAIN",
    "ORDER"
]

candidate_features = (
    basins
    .select(fields, None, False)
    .getInfo()["features"]
)

candidate_df = pd.DataFrame(
    [feature["properties"] for feature in candidate_features]
)

candidate_df = candidate_df.sort_values(
    by="SUB_AREA",
    ascending=False
).reset_index(drop=True)

display(candidate_df.round(2))

## 2.6 Geographic Verification of HydroBASINS Candidates

Calculate each candidate's centroid and geometric area to help
identify which polygons overlap the intended Pamba catchment.

In [ ]:
def add_geometry_metrics(feature):
    centroid = feature.geometry().centroid(maxError=100)
    coords = centroid.coordinates()

    return feature.set({
        "centroid_lon": coords.get(0),
        "centroid_lat": coords.get(1),
        "geometry_area_km2": feature.geometry().area(maxError=100).divide(1e6)
    })

basins_with_metrics = basins.map(add_geometry_metrics)

metric_fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "centroid_lon",
    "centroid_lat",
    "geometry_area_km2"
]

features = (
    basins_with_metrics
    .select(metric_fields, None, False)
    .getInfo()["features"]
)

geo_df = pd.DataFrame([
    feature["properties"] for feature in features
])

geo_df = geo_df.sort_values(
    "centroid_lat",
    ascending=False
).reset_index(drop=True)

display(geo_df.round(4))

### 2.7 Screen HydroBASINS Candidates by Geographic Extent

Compare each candidate's bounding coordinates with the official
Pamba basin map to identify polygons worth inspecting visually.

This is a preliminary screening step, not final boundary validation.

In [ ]:
def add_bbox_metrics(feature):
    ring = ee.List(
        feature.geometry()
        .bounds(maxError=100)
        .coordinates()
        .get(0)
    )

    lons = ring.map(lambda point: ee.List(point).get(0))
    lats = ring.map(lambda point: ee.List(point).get(1))

    return feature.set({
        "min_lon": ee.Number(lons.reduce(ee.Reducer.min())),
        "max_lon": ee.Number(lons.reduce(ee.Reducer.max())),
        "min_lat": ee.Number(lats.reduce(ee.Reducer.min())),
        "max_lat": ee.Number(lats.reduce(ee.Reducer.max()))
    })

basins_bbox = basins.map(add_bbox_metrics)

bbox_fields = [
    "HYBAS_ID", "PFAF_ID", "SUB_AREA",
    "min_lon", "max_lon", "min_lat", "max_lat"
]

bbox_features = (
    basins_bbox
    .select(bbox_fields, None, False)
    .getInfo()["features"]
)

bbox_df = pd.DataFrame([
    f["properties"] for f in bbox_features
])

display(
    bbox_df.sort_values("SUB_AREA")
    .reset_index(drop=True)
    .round(4)
)

## 2.7 Screen HydroBASINS Candidates by Geographic Extent

Compare each candidate's bounding coordinates with the official
Pamba basin map to identify polygons worth inspecting visually.

This is a preliminary screening step, not final boundary validation.

In [ ]:
def add_bbox_metrics(feature):
    ring = ee.List(
        feature.geometry()
        .bounds(maxError=100)
        .coordinates()
        .get(0)
    )

    lons = ring.map(lambda point: ee.List(point).get(0))
    lats = ring.map(lambda point: ee.List(point).get(1))

    return feature.set({
        "min_lon": ee.Number(lons.reduce(ee.Reducer.min())),
        "max_lon": ee.Number(lons.reduce(ee.Reducer.max())),
        "min_lat": ee.Number(lats.reduce(ee.Reducer.min())),
        "max_lat": ee.Number(lats.reduce(ee.Reducer.max()))
    })

basins_bbox = basins.map(add_bbox_metrics)

bbox_fields = [
    "HYBAS_ID", "PFAF_ID", "SUB_AREA",
    "min_lon", "max_lon", "min_lat", "max_lat"
]

bbox_features = (
    basins_bbox
    .select(bbox_fields, None, False)
    .getInfo()["features"]
)

bbox_df = pd.DataFrame([
    f["properties"] for f in bbox_features
])

display(
    bbox_df.sort_values("SUB_AREA")
    .reset_index(drop=True)
    .round(4)
)

## 2.8 Verify the HydroBASINS Dataset Level

Confirm the source collection and its metadata before selecting a catchment.

The current candidates have not yet been matched to the official Pamba River Basin boundary. Dataset-level verification helps avoid selecting polygons from an unintended HydroBASINS layer.

In [ ]:
# Inspect the collection source and candidate identifiers.
# This does not change or select any basin.

print("Candidate count:", basins.size().getInfo())

print("\nCandidate HYBAS_ID values:")
print(
    basins.aggregate_array("HYBAS_ID")
    .getInfo()
)

print("\nCandidate PFAF_ID values:")
print(
    basins.aggregate_array("PFAF_ID")
    .getInfo()
)

## 2.9 Compare HydroBASINS Levels

Compare Levels 6, 7, and 8 within the existing search area. For each level, inspect the five candidate catchments whose reported sub-basin areas are closest to the published Pamba basin area of approximately 2,235 km².

Area similarity is used only to narrow the candidates. Final selection requires geographic and boundary verification.

In [ ]:
REFERENCE_AREA_KM2 = 2235

level_results = []

for level in [6, 7, 8]:
    collection_id = (
        f"WWF/HydroSHEDS/v1/Basins/hybas_{level}"
    )

    candidates = (
        ee.FeatureCollection(collection_id)
        .filterBounds(search_area)
    )

    def add_comparison_metrics(feature):
        centroid = feature.geometry().centroid(
            maxError=100
        ).coordinates()

        area = ee.Number(feature.get("SUB_AREA"))

        return feature.set({
            "level": level,
            "area_difference_km2": area.subtract(
                REFERENCE_AREA_KM2
            ).abs(),
            "centroid_lon": centroid.get(0),
            "centroid_lat": centroid.get(1)
        })

    closest = (
        candidates
        .map(add_comparison_metrics)
        .sort("area_difference_km2")
        .limit(5)
        .select([
            "level",
            "HYBAS_ID",
            "PFAF_ID",
            "SUB_AREA",
            "UP_AREA",
            "area_difference_km2",
            "centroid_lon",
            "centroid_lat"
        ])
    )

    result = closest.getInfo()

    for feature in result["features"]:
        level_results.append(feature["properties"])

    print(
        f"Level {level}: "
        f"{candidates.size().getInfo()} candidates "
        "intersect the search area"
    )

level_df = pd.DataFrame(level_results)

display(
    level_df.sort_values(
        ["level", "area_difference_km2"]
    ).round(3).reset_index(drop=True)
)

## 2.10 Visually Screen Level 8 Catchment Boundaries

Display Level 8 HydroBASINS polygons intersecting the approximate
geographic extent of the official Pamba River Basin map.

The rectangular extent is for screening only. It is not the basin
boundary, and no candidate will be selected from this overlay alone.

In [ ]:
# Approximate extent read from the official basin map.
# This rectangle is NOT the Pamba basin boundary.

pamba_map_extent = ee.Geometry.Rectangle([
    76.20, 9.10, 77.25, 9.80
])

level8_candidates = (
    ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_8")
    .filterBounds(pamba_map_extent)
)

print(
    "Level 8 polygons intersecting the approximate map extent:",
    level8_candidates.size().getInfo()
)

Map.centerObject(pamba_map_extent, 9)

Map.addLayer(
    pamba_map_extent,
    {"color": "yellow"},
    "Approximate official-map extent"
)

Map.addLayer(
    level8_candidates.style(
        color="cyan",
        fillColor="00000000",
        width=2
    ),
    {},
    "HydroBASINS Level 8 candidates"
)

Map

## 2.11 Label HydroBASINS Candidates for Identification

Display the Level 8 candidate catchments with their identifiers and
reported sub-basin areas to connect the map view with the attribute table.

Use the labels to inspect spatial relationships. Do not treat partial
overlap with the official Pamba map as boundary confirmation.

In [ ]:
# Prepare concise labels for the candidate polygons.
def add_candidate_label(feature):
    label = ee.String(
        ee.Number(feature.get("HYBAS_ID")).format()
    ).cat(" | ").cat(
        ee.Number(feature.get("SUB_AREA")).format("%.0f")
    ).cat(" km²")

    return feature.set("map_label", label)

labeled_level8 = level8_candidates.map(add_candidate_label)

# Add candidate outlines and labels.
Map.addLayer(
    labeled_level8.style(
        color="cyan",
        fillColor="00000000",
        width=2
    ),
    {},
    "Level 8 candidate boundaries"
)

Map.addLayer(
    labeled_level8.style(
        color="white",
        pointSize=1
    ),
    {},
    "Candidate label anchors"
)

Map

## 2.12 Tabulate Level 8 Candidates Within the Pamba Map Extent

List the Level 8 catchments intersecting the approximate extent of the
official Pamba basin map, including their IDs, areas, and centroids.

Use this table to narrow the candidates for further geographic
verification. Do not select a basin solely by area or centroid.

In [ ]:
# Reuse pamba_map_extent and level8_candidates from Section 2.10.

def candidate_summary(feature):
    centroid = feature.geometry().centroid(
        maxError=100
    ).coordinates()

    return feature.set({
        "centroid_lon": centroid.get(0),
        "centroid_lat": centroid.get(1),
        "geometry_area_km2": (
            feature.geometry().area(maxError=100).divide(1e6)
        )
    })

summary_fc = level8_candidates.map(candidate_summary)

summary_fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "centroid_lat",
    "centroid_lon",
    "geometry_area_km2"
]

summary_info = (
    summary_fc
    .select(summary_fields, None, False)
    .getInfo()
)

summary_df = pd.DataFrame([
    feature["properties"]
    for feature in summary_info["features"]
])

summary_df = summary_df.sort_values(
    "centroid_lat"
).reset_index(drop=True)

print("Level 8 candidates:", len(summary_df))
display(summary_df.round(4))

## 2.13 Inspect HydroBASINS Drainage Connectivity

Update the approximate screening extent using the coordinate range
shown on the official Pamba River Basin map.

Inspect Level 8 catchments together with their downstream links,
main-basin identifiers, reported areas, and geographic positions.

The purpose is to understand how candidate catchments connect
hydrologically before attempting to identify or assemble the basin boundary.

In [ ]:
# Broad screening extent based on the official map's coordinate grid.
# This rectangle is NOT the Pamba basin boundary.

pamba_screening_extent = ee.Geometry.Rectangle([
    76.20, 8.85,
    77.25, 9.70
])

level8_screen = (
    ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_8")
    .filterBounds(pamba_screening_extent)
)

def add_candidate_metrics(feature):
    centroid = feature.geometry().centroid(
        maxError=100
    ).coordinates()

    geometry_area = (
        feature.geometry()
        .area(maxError=100)
        .divide(1e6)
    )

    return feature.set({
        "centroid_lon": centroid.get(0),
        "centroid_lat": centroid.get(1),
        "geometry_area_km2": geometry_area
    })

level8_screen_metrics = level8_screen.map(
    add_candidate_metrics
)

connectivity_fields = [
    "HYBAS_ID",
    "PFAF_ID",
    "SUB_AREA",
    "UP_AREA",
    "NEXT_DOWN",
    "MAIN_BAS",
    "NEXT_SINK",
    "COAST",
    "ORDER",
    "DIST_MAIN",
    "DIST_SINK",
    "centroid_lat",
    "centroid_lon",
    "geometry_area_km2"
]

connectivity_info = (
    level8_screen_metrics
    .select(connectivity_fields, None, False)
    .getInfo()
)

connectivity_df = pd.DataFrame([
    feature["properties"]
    for feature in connectivity_info["features"]
])

connectivity_df = connectivity_df.sort_values(
    ["centroid_lat", "centroid_lon"]
).reset_index(drop=True)

print(
    "Level 8 candidates in updated screening extent:",
    len(connectivity_df)
)

display(connectivity_df.round(4))

## 2.14 Inspect an Official Hydrological Boundary Dataset

Load the downloaded government sub-basin boundary archive into Colab
and inspect its contents before processing any geometry.

The dataset must be checked for its source, coordinate reference system,
attribute fields, and geographic coverage before use.

In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_files = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_files:
    raise ValueError(
        "Please upload the downloaded boundary dataset as a ZIP file."
    )

boundary_zip = zip_files[0]
extract_dir = "/content/official_boundary_data"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(boundary_zip, "r") as archive:
    archive.extractall(extract_dir)
    archive_contents = archive.namelist()

print("Archive:", boundary_zip)
print("Extracted files:")

for name in archive_contents:
    print("-", name)

## 2.15 Load and Inspect Official Sub-Basin Boundaries

Load the extracted shapefile with GeoPandas and inspect its coordinate reference system, attribute fields, feature count, and sample records. These checks will help us identify sub-basins using their attributes and geographic locations before selecting the Pamba River Basin boundary.


In [ ]:
import geopandas as gpd
import os

shapefile_path = os.path.join(
    extract_dir,
    "Subbasin.shp"
)

if not os.path.exists(shapefile_path):
    raise FileNotFoundError(
        f"Shapefile not found at: {shapefile_path}"
    )

subbasins = gpd.read_file(shapefile_path)

print("Feature count:", len(subbasins))
print("CRS:", subbasins.crs)
print("\nAttribute columns:")
print(subbasins.columns.tolist())

print("\nFirst five records:")
display(subbasins.drop(columns="geometry").head())

print("\nGeometry types:")
print(subbasins.geometry.geom_type.value_counts())

print("\nTotal bounds (minx, miny, maxx, maxy):")
print(subbasins.total_bounds)


## 2.16 Search Official Sub-Basin Names

Inspect all basin and sub-basin names, then filter for records containing likely matches for Pamba, Kerala, and nearby regional names. This is an initial text-based search; absence of a direct “Pamba” match does not mean the dataset lacks the relevant boundary.


In [ ]:
# Convert a copy to WGS84 for geographic inspection
subbasins_wgs84 = subbasins.to_crs(epsg=4326)

# Display all official basin and sub-basin names
name_table = (
    subbasins_wgs84[
        ["objectid", "sbcode", "bacode", "ba_name", "sub_basin"]
    ]
    .sort_values(["ba_name", "sub_basin"])
    .reset_index(drop=True)
)

display(name_table)

# Search likely names without assuming Pamba is explicitly listed
keywords = r"pamba|pampa|kerala|periyar|bharathapuzha|west flowing"

matches = name_table[
    name_table["ba_name"].str.contains(
        keywords, case=False, na=False, regex=True
    )
    | name_table["sub_basin"].str.contains(
        keywords, case=False, na=False, regex=True
    )
]

print("\nPotentially relevant records:")
display(matches)

print("\nGeographic bounds by candidate (longitude/latitude):")
if not matches.empty:
    bounds = subbasins_wgs84.loc[
        matches.index, ["objectid", "sub_basin", "geometry"]
    ].copy()
    bounds[["min_lon", "min_lat", "max_lon", "max_lat"]] = (
        bounds.geometry.bounds
    )
    display(bounds.drop(columns="geometry"))


## 2.17 Verify the Official Pamba Sub-Basin Geometry

Select the official feature using its unique `sbcode` and inspect its attributes, geographic bounds, area, and geometry validity. This verifies that the correct feature was selected before using it as the study-area boundary.

In [ ]:
# Select the Pamba sub-basin using its official code
pamba_official = subbasins_wgs84[
    subbasins_wgs84["sbcode"].astype(str).str.upper() == "PAM"
].copy()

if pamba_official.empty:
    raise ValueError("No feature with sbcode='PAM' was found.")

# Report the identifying attributes
display(
    pamba_official[
        ["objectid", "sbconc", "sbcode", "bacode",
         "ba_name", "sub_basin", "shape_Area"]
    ]
)

# Inspect geometry and geographic extent
print("Feature count:", len(pamba_official))
print("CRS:", pamba_official.crs)
print("Geometry type:", pamba_official.geometry.geom_type.tolist())
print("Geometry valid:", pamba_official.geometry.is_valid.tolist())
print("Bounds (min longitude, min latitude, max longitude, max latitude):")
print(pamba_official.total_bounds)

# Calculate area after projecting to the dataset's original metric CRS
pamba_metric = pamba_official.to_crs(subbasins.crs)
area_km2 = pamba_metric.geometry.area.sum() / 1_000_000

print(f"Calculated geometry area: {area_km2:,.1f} km²")


## 2.18 Diagnose the Official Boundary Geometry

Check the invalid geometry reason, the number of polygon components, and the feature's area in its original projected CRS. Also inspect the other sub-basins belonging to the same larger basin. This helps determine whether the record represents a broad regional grouping rather than the Pamba catchment itself.


In [ ]:
from shapely.validation import explain_validity

feature = pamba_official.iloc[0]
geom = feature.geometry

print("Official sub-basin:", feature["sub_basin"])
print("Parent basin:", feature["ba_name"])
print("Validity reason:", explain_validity(geom))
print("Geometry type:", geom.geom_type)
print("Number of polygon components:",
      len(geom.geoms) if geom.geom_type == "MultiPolygon" else 1)
print("Geographic bounds:", geom.bounds)

# Inspect all records assigned to the same parent basin
parent_name = feature["ba_name"]

same_parent = subbasins_wgs84[
    subbasins_wgs84["ba_name"] == parent_name
].copy()

same_parent["area_km2"] = (
    same_parent.to_crs(subbasins.crs).geometry.area / 1_000_000
)

print("\nFeatures in the same parent basin:")
display(
    same_parent[
        ["objectid", "sbcode", "sub_basin", "area_km2"]
    ].sort_values("area_km2", ascending=False)
)

## 2.19 Narrow HydroBASINS Candidates to the Pamba Region

Filter Level 8 HydroBASINS polygons intersecting a tighter geographic window around the expected Pamba basin. Print their identifiers, sub-basin areas, upstream areas, and geographic bounds. These candidates will be reviewed spatially before selecting a basin or delineating a catchment from a DEM.

In [ ]:
import ee
import pandas as pd

# Approximate Pamba-region screening window only — not the basin boundary
pamba_screening_extent = ee.Geometry.Rectangle(
    [76.20, 8.85, 77.25, 9.70]
)

# Reuse the existing Level 8 HydroBASINS collection: basins
# Filter polygons intersecting the screening window
regional_candidates = basins.filterBounds(pamba_screening_extent)

candidate_info = regional_candidates.map(
    lambda f: f.set(
        "centroid",
        f.geometry().centroid(100).coordinates()
    )
)

features = candidate_info.getInfo()["features"]

rows = []
for item in features:
    p = item["properties"]
    centroid = p.get("centroid", [None, None])
    geometry = item.get("geometry", {})
    coords = geometry.get("coordinates", [])

    rows.append({
        "HYBAS_ID": p.get("HYBAS_ID"),
        "PFAF_ID": p.get("PFAF_ID"),
        "SUB_AREA_km2": p.get("SUB_AREA"),
        "UP_AREA_km2": p.get("UP_AREA"),
        "COAST": p.get("COAST"),
        "ORDER": p.get("ORDER"),
        "centroid_lon": centroid[0] if len(centroid) > 0 else None,
        "centroid_lat": centroid[1] if len(centroid) > 1 else None,
    })

candidate_table = pd.DataFrame(rows)

if candidate_table.empty:
    print("No Level 8 candidates intersect this window.")
else:
    candidate_table = candidate_table.sort_values(
        ["SUB_AREA_km2", "UP_AREA_km2"],
        ascending=False
    ).reset_index(drop=True)

    print("Candidates intersecting screening window:",
          len(candidate_table))
    display(candidate_table)

## 2.20 Visualize HydroBASINS Candidates

Display the 12 Level 8 polygons intersecting the Pamba-region screening window. Label each polygon with its HydroBASINS identifier and sub-area. The map is for visual screening only; the rectangular search window is not the study boundary.


In [ ]:
import geemap

# Add labels and display properties to the candidate features
def add_candidate_label(feature):
    label = (
        ee.Number(feature.get("HYBAS_ID")).format()
        .cat(" | ")
        .cat(ee.Number(feature.get("SUB_AREA")).format("%.0f"))
        .cat(" km²")
    )
    return feature.set("map_label", label)

labeled_candidates = regional_candidates.map(add_candidate_label)

# Create a fresh map
candidate_map = geemap.Map(basemap="Esri.WorldTopoMap")
candidate_map.centerObject(pamba_screening_extent, 8)

candidate_map.addLayer(
    pamba_screening_extent,
    {"color": "yellow"},
    "Screening window"
)

candidate_map.addLayer(
    labeled_candidates.style(
        color="00FFFF",
        fillColor="00000000",
        width=2
    ),
    {},
    "HydroBASINS candidate boundaries"
)

# Add centroid points to help associate map locations with the table
candidate_centroids = labeled_candidates.map(
    lambda f: ee.Feature(
        f.geometry().centroid(100),
        {
            "HYBAS_ID": f.get("HYBAS_ID"),
            "SUB_AREA": f.get("SUB_AREA")
        }
    )
)

candidate_map.addLayer(
    candidate_centroids,
    {"color": "red", "pointSize": 5},
    "Candidate centroids"
)

candidate_map

## 2.21 Inspect Candidate Drainage Units and Their Outlets

Inspect the HydroBASINS polygons individually and compare their locations with the Pamba River and its downstream outlet. This is a preliminary step to help identify a plausible outlet location before DEM-based watershed delineation.


In [ ]:
# Display the candidate table ordered by location and area
display(
    candidate_table[
        [
            "HYBAS_ID",
            "SUB_AREA_km2",
            "UP_AREA_km2",
            "COAST",
            "ORDER",
            "centroid_lon",
            "centroid_lat",
        ]
    ].sort_values(["centroid_lat", "centroid_lon"])
)

print(
    "\nReference area range: approximately 2,235–2,487 km²."
)
print(
    "Reminder: candidate area similarity alone does not establish "
    "that a polygon represents the Pamba catchment."
)

## 2.25 Load and Validate the Initial Watershed Delineation

Load the GeoJSON exported from the watershed tool. Separate the watershed polygon from the river-network features, verify geometry validity and geographic extent, and independently calculate the watershed area in a suitable projected coordinate system.


In [ ]:
import geopandas as gpd
import pandas as pd
from google.colab import files

uploaded = files.upload()

geojson_files = [
    name for name in uploaded
    if name.lower().endswith((".geojson", ".json"))
]

if not geojson_files:
    raise ValueError("Upload the exported watershed GeoJSON file.")

watershed_file = geojson_files[0]
delineation = gpd.read_file(watershed_file)

print("File:", watershed_file)
print("CRS:", delineation.crs)
print("Total features:", len(delineation))
print("\nGeometry types:")
print(delineation.geometry.geom_type.value_counts())

# Separate polygon features from river-network lines
watershed_polygon = delineation[
    delineation.geometry.geom_type.isin(
        ["Polygon", "MultiPolygon"]
    )
].copy()

river_lines = delineation[
    delineation.geometry.geom_type.isin(
        ["LineString", "MultiLineString"]
    )
].copy()

if watershed_polygon.empty:
    raise ValueError("No watershed polygon found in the GeoJSON.")

# Calculate area using a global equal-area CRS
watershed_equal_area = watershed_polygon.to_crs(epsg=6933)
calculated_area_km2 = (
    watershed_equal_area.geometry.area.sum() / 1_000_000
)

print("\nWatershed polygon count:", len(watershed_polygon))
print("River-network feature count:", len(river_lines))
print("Geometry valid:", watershed_polygon.geometry.is_valid.tolist())
print("Bounds (min lon, min lat, max lon, max lat):")
print(watershed_polygon.to_crs(epsg=4326).total_bounds)
print(f"Calculated watershed area: {calculated_area_km2:,.1f} km²")


## 2.26 Visually Inspect the Delineated Watershed

Overlay the exported watershed boundary, its river-network lines, and the selected outlet location on a topographic basemap. Inspect whether the outlet connects to the main river and whether the boundary appears to include neighbouring catchments.


In [ ]:
import geemap
import ee

# Selected point used in the watershed tool: longitude, latitude
outlet = ee.Geometry.Point([76.4468, 9.3418])

# Create the inspection map
check_map = geemap.Map(basemap="Esri.WorldTopoMap")
check_map.centerObject(outlet, 9)

# Add the exported watershed polygon
check_map.add_gdf(
    watershed_polygon.to_crs(epsg=4326),
    layer_name="Delineated watershed",
    style={"color": "red", "fillColor": "00000000", "weight": 3}
)

# Add the river-network lines from the exported GeoJSON
if not river_lines.empty:
    check_map.add_gdf(
        river_lines.to_crs(epsg=4326),
        layer_name="Exported river network",
        style={"color": "blue", "weight": 2}
    )

# Add the selected outlet marker
check_map.addLayer(
    outlet,
    {"color": "yellow"},
    "Selected outlet point"
)

check_map

## Step 2.30 — Inspect finer HydroBASINS candidates

**Objective:** Identify smaller hydrological catchments around the Pamba River Basin and compare their areas and locations with the official Kerala basin map.

**Method:** Query HydroBASINS level 9, filter catchments intersecting the Pamba search region, and calculate each candidate's approximate area and centroid.

**Important limitation:** A candidate's area being close to a published Pamba basin area does not prove that its boundary is correct. We will inspect its shape and river connections before accepting it.

In [ ]:
import ee
import geemap
import pandas as pd
from IPython.display import display

# Broad search window around the Pamba basin.
# This is a search extent, NOT the final basin boundary.
pamba_search_area = ee.Geometry.Rectangle(
    [76.15, 9.00, 77.45, 9.85]
)

# HydroBASINS level 9: finer subdivisions than the level 8
# candidates examined earlier.
basins9 = ee.FeatureCollection(
    "WWF/HydroSHEDS/v1/Basins/hybas_9"
)

candidates9 = basins9.filterBounds(pamba_search_area)

# Add approximate area and centroid coordinates.
def add_candidate_metrics(feature):
    centroid = feature.geometry().centroid(1000).coordinates()
    area_km2 = feature.geometry().area(1000).divide(1_000_000)

    return feature.set({
        "area_km2": area_km2,
        "centroid_lonlat": centroid
    })

candidates9_metrics = candidates9.map(add_candidate_metrics)

# Retrieve only attributes, not full geometries.
candidate_data = candidates9_metrics.select([
    "HYBAS_ID",
    "NEXT_DOWN",
    "MAIN_BAS",
    "UP_AREA",
    "SUB_AREA",
    "area_km2",
    "centroid_lonlat"
]).getInfo()

rows = [
    feature["properties"]
    for feature in candidate_data["features"]
]

candidates_df = pd.DataFrame(rows)

if candidates_df.empty:
    print("No level-9 catchments intersect the search extent.")
else:
    candidates_df = candidates_df.sort_values(
        "area_km2", ascending=False
    ).reset_index(drop=True)

    print("Level-9 candidate count:", len(candidates_df))
    display(candidates_df)

    # Map the search region and candidate catchments.
    candidate_map = geemap.Map(basemap="Esri.WorldTopoMap")
    candidate_map.centerObject(pamba_search_area, 9)

    candidate_map.addLayer(
        pamba_search_area,
        {"color": "yellow"},
        "Pamba search extent"
    )

    candidate_map.addLayer(
        candidates9.style(
            color="red",
            fillColor="00000000",
            width=2
        ),
        {},
        "HydroBASINS level-9 candidates"
    )

    candidate_map

## Step 2.31 — Inspect the candidate drainage group

**Objective:** Examine catchments associated with HydroBASINS group `4090029720` and compare their combined spatial pattern with the official Pamba basin map.

**Method:** Highlight the group on the map and inspect its individual catchments. The group is a candidate for investigation, not an accepted Pamba boundary.

**Decision rule:** Do not accept the group unless its shape, river connections, and basin definition are consistent with the official reference.

In [ ]:
# Candidate group identified in the previous table.
target_main_basin = 4090029720

# Select catchments associated with this drainage group.
pamba_group = candidates9.filter(
    ee.Filter.eq("MAIN_BAS", target_main_basin)
)

# Identify the individual feature with this HYBAS_ID, if present.
group_root = candidates9.filter(
    ee.Filter.eq("HYBAS_ID", target_main_basin)
)

print("Catchments in candidate group:", pamba_group.size().getInfo())
print("Feature matching group ID:", group_root.size().getInfo())

# Summarize the group's catchments.
group_info = pamba_group.select([
    "HYBAS_ID",
    "NEXT_DOWN",
    "MAIN_BAS",
    "SUB_AREA",
    "UP_AREA"
]).getInfo()

group_rows = [
    feature["properties"]
    for feature in group_info["features"]
]

group_df = pd.DataFrame(group_rows)

if not group_df.empty:
    display(
        group_df.sort_values("SUB_AREA", ascending=False)
        .reset_index(drop=True)
    )

# Map the group against the wider candidate network.
group_map = geemap.Map(basemap="Esri.WorldTopoMap")
group_map.centerObject(pamba_search_area, 9)

group_map.addLayer(
    pamba_search_area,
    {"color": "yellow"},
    "Search extent"
)

group_map.addLayer(
    candidates9.style(
        color="777777",
        fillColor="00000000",
        width=1
    ),
    {},
    "All level-9 candidates"
)

group_map.addLayer(
    pamba_group.style(
        color="red",
        fillColor="FF000020",
        width=2
    ),
    {},
    "Candidate group 4090029720"
)

group_map.addLayer(
    group_root.style(
        color="orange",
        fillColor="FFA50030",
        width=3
    ),
    {},
    "Feature matching group ID"
)

group_map

## Step 2.32 — Establish the official Pamba basin reference

**Objective:** Use the Kerala Irrigation Department's Pamba basin map to identify the basin's geographic extent and distinguish the Pamba-only basin from neighbouring catchments.

**Method:** Review the official map and compare its boundary with the HydroBASINS candidates. We will not accept a candidate based on area alone.

**Expected outcome:** A documented decision on whether a suitable existing GIS boundary is available or whether the official map needs to be georeferenced and digitized.

## Step 2.33 — Inspect the official Pamba basin map

**Objective:** Inspect the original Kerala Irrigation Department map before georeferencing it.

**Checks:** Image dimensions, colour mode, coordinate-grid labels, and visible basin-boundary information.

**Important:** This is a reference map, not yet a georeferenced GIS boundary. We must account for the map's coordinate grid and projection before digitizing the basin outline.

In [ ]:
from google.colab import files
from PIL import Image
from IPython.display import display
import os

uploaded_map = files.upload()

image_files = [
    name for name in uploaded_map
    if name.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))
]

if not image_files:
    raise ValueError("Upload the original downloaded PAMBA map image.")

map_path = image_files[0]

with Image.open(map_path) as img:
    print("File:", os.path.basename(map_path))
    print("Image dimensions (width, height):", img.size)
    print("Image format:", img.format)
    print("Colour mode:", img.mode)

    display(img.copy())

## Step 2.34 — Inspect coordinate ticks and map frame

**Objective:** Identify the longitude and latitude ticks around the official Pamba basin map.

**Method:** Display enlarged crops of the map's four margins so the coordinate labels, tick marks, and map frame can be inspected.

**Why:** These control points will help us establish the image-to-coordinate transformation. We will not digitize the basin until the reference points are understood.

In [ ]:
from PIL import Image
from IPython.display import display

img = Image.open("PAMBA.jpg").convert("RGB")
width, height = img.size

print(f"Image size: {width} × {height} pixels")

# Enlarged strips containing the coordinate labels and map frame.
margin_crops = {
    "Top margin — longitude ticks": (0, 0, width, int(height * 0.12)),
    "Bottom margin — longitude ticks": (
        0, int(height * 0.88), width, height
    ),
    "Left margin — latitude ticks": (0, 0, int(width * 0.10), height),
    "Right margin — latitude ticks": (
        int(width * 0.90), 0, width, height
    ),
}

for title, crop_box in margin_crops.items():
    print("\n" + title)
    crop = img.crop(crop_box)
    display(crop)

## Step 2.35 — Establish pixel coordinates for georeferencing

**Objective:** Display the original map with pixel-coordinate axes so longitude and latitude tick positions can be measured consistently.

**Method:** Plot the image in its original pixel coordinate system, with the origin at the top-left corner.

**Output:** A reference image for recording pixel positions of the printed geographic grid ticks. These control points will be used to test the georeferencing transformation before digitizing the basin boundary.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

img = Image.open("PAMBA.jpg").convert("RGB")
image_array = np.array(img)
height, width = image_array.shape[:2]

fig, ax = plt.subplots(figsize=(18, 12))

ax.imshow(image_array, origin="upper")
ax.set_xlim(0, width)
ax.set_ylim(height, 0)

# Pixel-coordinate ticks every 200 pixels.
ax.set_xticks(np.arange(0, width + 1, 200))
ax.set_yticks(np.arange(0, height + 1, 200))

ax.grid(True, linewidth=0.4, alpha=0.45)
ax.set_xlabel("Pixel X (from left edge)")
ax.set_ylabel("Pixel Y (from top edge)")
ax.set_title("Official Pamba Basin Map — Pixel Coordinate Reference")

plt.tight_layout()
plt.show()

## Step 2.36 — Measure geographic control points

**Objective:** Locate the printed longitude and latitude tick marks in the original map image.

**Method:** Extract the top and left margins at native resolution and display pixel-coordinate axes. Use the printed geographic labels to associate image positions with known coordinates.

**Quality check:** We will use multiple control points and check alignment before accepting the georeferencing. Approximate coordinates are provisional until verified.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

img = Image.open("PAMBA.jpg").convert("RGB")
arr = np.array(img)
height, width = arr.shape[:2]

# Inspect native-resolution margins.
fig, axes = plt.subplots(2, 1, figsize=(18, 8))

# Top margin: longitude labels and tick marks.
top_end = int(height * 0.10)
axes[0].imshow(arr[:top_end, :, :], origin="upper")
axes[0].set_xlim(0, width)
axes[0].set_ylim(top_end, 0)
axes[0].set_xticks(np.arange(0, width + 1, 100))
axes[0].grid(True, alpha=0.35)
axes[0].set_title("Top margin — native pixel coordinates")
axes[0].set_xlabel("Original image X (pixels)")
axes[0].set_ylabel("Original image Y (pixels)")

# Left margin: latitude labels and tick marks.
left_end = int(width * 0.12)
axes[1].imshow(arr[:, :left_end, :], origin="upper")
axes[1].set_xlim(0, left_end)
axes[1].set_ylim(height, 0)
axes[1].set_xticks(np.arange(0, left_end + 1, 50))
axes[1].set_yticks(np.arange(0, height + 1, 100))
axes[1].grid(True, alpha=0.35)
axes[1].set_title("Left margin — native pixel coordinates")
axes[1].set_xlabel("Original image X (pixels)")
axes[1].set_ylabel("Original image Y (pixels)")

plt.tight_layout()
plt.show()

## Step 2.37 — Inspect latitude control points at readable scale

**Objective:** Enlarge the left map margin without distorting its aspect ratio so the latitude tick marks can be located accurately.

**Method:** Display the left margin in a tall figure and the top margin separately. Keep the original image pixel coordinates on both axes.

**Quality check:** Do not record approximate pixel positions from the compressed figure. We will establish the control points from the corrected views.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

img = Image.open("PAMBA.jpg").convert("RGB")
arr = np.array(img)
height, width = arr.shape[:2]

# Left margin: tall figure preserves the original aspect ratio.
left_end = 420
fig, ax = plt.subplots(figsize=(6, 18))
ax.imshow(arr[:, :left_end, :], origin="upper", aspect="auto")
ax.set_xlim(0, left_end)
ax.set_ylim(height, 0)
ax.set_xticks(np.arange(0, left_end + 1, 50))
ax.set_yticks(np.arange(0, height + 1, 100))
ax.grid(True, alpha=0.35)
ax.set_title("Left margin — latitude ticks")
ax.set_xlabel("Original image X (pixels)")
ax.set_ylabel("Original image Y (pixels)")
plt.tight_layout()
plt.show()

# Top margin: show longitude ticks separately.
top_end = 230
fig, ax = plt.subplots(figsize=(18, 4))
ax.imshow(arr[:top_end, :, :], origin="upper", aspect="auto")
ax.set_xlim(0, width)
ax.set_ylim(top_end, 0)
ax.set_xticks(np.arange(0, width + 1, 100))
ax.set_yticks(np.arange(0, top_end + 1, 25))
ax.grid(True, alpha=0.35)
ax.set_title("Top margin — longitude ticks")
ax.set_xlabel("Original image X (pixels)")
ax.set_ylabel("Original image Y (pixels)")
plt.tight_layout()
plt.show()

## Step 2.38 — Inspect the Official Pamba Basin Map in Pixel Coordinates

**Objective:** Prepare the original high-resolution `PAMBA.jpg` for georeferencing by identifying the pixel positions of its printed longitude and latitude ticks.

**Why this matters:** We need to establish a reliable relationship between image pixels and geographic coordinates before converting the official basin boundary into a GIS polygon.

**Procedure:**
1. Load the original `PAMBA.jpg` without resizing it.
2. Display the complete map with its original pixel-coordinate axes.
3. Display enlarged crops of the top and left margins, where the longitude and latitude labels are printed.
4. Use the pixel axes to identify tick-mark positions in the next step.

**Important:** Do not digitize the basin boundary or calculate its geographic area yet. First, verify the coordinate reference using several clearly visible ticks along both margins.


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

# Locate the original, full-resolution official map.
image_path = Path("PAMBA.jpg")

if not image_path.exists():
    uploaded = files.upload()
    candidates = [
        name for name in uploaded
        if name.lower().endswith((".jpg", ".jpeg"))
    ]

    if not candidates:
        raise FileNotFoundError(
            "Please upload the original official PAMBA.jpg image."
        )

    image_path = Path(candidates[0])

image = Image.open(image_path).convert("RGB")
width, height = image.size

print("File:", image_path.name)
print(f"Original dimensions: {width} × {height} pixels")
print("Pixel origin: top-left corner (x=0, y=0)")

# Approximate margin crops, retaining original pixel coordinates.
top_crop_end = int(height * 0.12)
left_crop_end = int(width * 0.15)

fig, axes = plt.subplots(3, 1, figsize=(18, 18))

# 1. Complete map.
axes[0].imshow(image)
axes[0].set_title("Official Pamba Basin Map — Full Image")
axes[0].set_xlabel("Original image X (pixels)")
axes[0].set_ylabel("Original image Y (pixels)")
axes[0].set_xticks(range(0, width + 1, 200))
axes[0].set_yticks(range(0, height + 1, 200))
axes[0].grid(True, alpha=0.25)

# 2. Top margin: longitude ticks.
axes[1].imshow(image.crop((0, 0, width, top_crop_end)))
axes[1].set_title("Top Margin — Longitude Tick Positions")
axes[1].set_xlabel("Original image X (pixels)")
axes[1].set_ylabel("Original image Y (pixels)")
axes[1].set_xticks(range(0, width + 1, 100))
axes[1].set_yticks(range(0, top_crop_end + 1, 25))
axes[1].grid(True, alpha=0.35)

# 3. Left margin: latitude ticks.
axes[2].imshow(image.crop((0, 0, left_crop_end, height)))
axes[2].set_title("Left Margin — Latitude Tick Positions")
axes[2].set_xlabel("Original image X (pixels)")
axes[2].set_ylabel("Original image Y (pixels)")
axes[2].set_xticks(range(0, left_crop_end + 1, 50))
axes[2].set_yticks(range(0, height + 1, 100))
axes[2].grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

## Step 2.39 — Prepare the Official Pamba Basin Map for Georeferencing

**Objective:** Prepare the official Kerala Irrigation Department `PAMBA.jpg` map for georeferencing using its printed latitude and longitude ticks.

We will use the original full-resolution image. The map's printed coordinate grid provides reference points that can connect image pixel positions to geographic coordinates.

**Important limitation:** The map's basin boundary appears to follow administrative boundaries in places. Georeferencing will establish its geographic position; it will not convert that boundary into a purely terrain-derived watershed.

**Expected output:** The original image loaded successfully, with its dimensions confirmed, ready for control-point selection.

In [ ]:
from pathlib import Path
from PIL import Image
from google.colab import files
from IPython.display import display

# Reuse the existing file if it is already present in Colab.
map_path = Path("/content/PAMBA.jpg")

if not map_path.exists():
    uploaded = files.upload()
    matching_files = [
        name for name in uploaded
        if name.lower().endswith((".jpg", ".jpeg"))
    ]

    if not matching_files:
        raise FileNotFoundError(
            "Upload the original full-resolution PAMBA.jpg map."
        )

    map_path = Path("/content") / matching_files[0]

official_map = Image.open(map_path).convert("RGB")

print("Map file:", map_path.name)
print("Image dimensions (width, height):", official_map.size)
print("Image format:", "RGB")
display(official_map.resize((1100, int(1100 * official_map.height / official_map.width))))


## Step 2.40 — Identify Geographic Control Points

**Objective:** Establish image-pixel positions corresponding to known longitude and latitude values printed on the official Pamba Basin map.

We will use coordinate ticks along the map frame, distributed across the image, rather than estimating coordinates from the basin outline.

The original map shows longitude ticks at 5-minute intervals and latitude ticks at 5-minute intervals. We will record their pixel positions and geographic coordinates, then assess the fit before exporting a georeferenced raster.

**Important:** Do not use pixel positions measured from a resized preview. All pixel coordinates must refer to the original 3306 × 2340 image.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Use the original image loaded in Step 2.39.
img = np.asarray(official_map)
height, width = img.shape[:2]

fig, ax = plt.subplots(figsize=(18, 12))
ax.imshow(img)
ax.set_title("Step 2.40 — Original PAMBA.jpg with Pixel Coordinates")
ax.set_xlabel("Pixel X (from left edge)")
ax.set_ylabel("Pixel Y (from top edge)")

# Display a light grid to help estimate tick-mark pixel positions.
ax.set_xticks(np.arange(0, width + 1, 100))
ax.set_yticks(np.arange(0, height + 1, 100))
ax.grid(True, linewidth=0.35, alpha=0.35)
ax.set_xlim(0, width - 1)
ax.set_ylim(height - 1, 0)

plt.tight_layout()
plt.show()

print(f"Original image dimensions: {width} × {height} pixels")


## Step 2.41 — Record Map Coordinate Control Points

**Objective:** Create a set of pixel-to-geographic control points from the coordinate ticks printed around the official map.

The map uses 5-minute intervals:
- Longitude: 76°20′E, 76°25′E, …, 77°15′E
- Latitude: 9°45′N, 9°40′N, …, 9°05′N

We will begin with approximate pixel positions for the ticks along the top and left edges. These estimates must be checked against the original image before georeferencing. We will use multiple points to estimate a transformation and inspect its residual errors.

**Important:** This map may contain slight scan/layout distortions. If a simple affine transformation does not fit well, we will assess the residuals before choosing another method.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Approximate tick locations read from the displayed original image.
# Verify marker placement visually before using these as control points.
longitude_ticks = [
    (76 + 20/60, 170),
    (76 + 25/60, 320),
    (76 + 30/60, 470),
    (76 + 35/60, 620),
    (76 + 40/60, 770),
    (76 + 45/60, 920),
    (76 + 50/60, 1070),
    (76 + 55/60, 1220),
    (77 + 0/60, 1370),
    (77 + 5/60, 1520),
    (77 + 10/60, 1670),
    (77 + 15/60, 1820),
]

latitude_ticks = [
    (9 + 45/60, 105),
    (9 + 40/60, 350),
    (9 + 35/60, 600),
    (9 + 30/60, 850),
    (9 + 25/60, 1100),
    (9 + 20/60, 1350),
    (9 + 15/60, 1600),
    (9 + 10/60, 1850),
    (9 + 5/60, 2100),
]

fig, ax = plt.subplots(figsize=(16, 10))
ax.imshow(official_map)

for lon, x in longitude_ticks:
    ax.plot(x, 95, "ro", markersize=4)
    ax.text(x, 75, f"{lon:.4f}", color="red",
            fontsize=7, rotation=45, ha="left")

for lat, y in latitude_ticks:
    ax.plot(110, y, "bo", markersize=4)
    ax.text(120, y, f"{lat:.4f}", color="blue",
            fontsize=7, va="bottom")

ax.set_xlim(50, 1900)
ax.set_ylim(2200, 30)
ax.set_title("Step 2.41 — Check Approximate Coordinate Tick Positions")
ax.set_xlabel("Pixel X")
ax.set_ylabel("Pixel Y")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Step 2.42 — Inspect Coordinate Tick Positions Along the Map Frame

**Objective:** Identify the actual pixel positions of the printed longitude and latitude ticks.

The previous overlay used rough estimates, so those positions must not be used to georeference the raster. This step isolates the top and left map margins at full resolution so the tick marks can be measured more reliably.

**Expected output:** Enlarged views of the top and left margins with pixel coordinates. We will use the measured positions to build a verified control-point table before fitting any transformation.

In [ ]:
import matplotlib.pyplot as plt

# Original image, not the resized preview
width, height = official_map.size

fig, axes = plt.subplots(2, 1, figsize=(18, 9))

# Top margin: longitude labels and tick marks
axes[0].imshow(official_map)
axes[0].set_xlim(80, width - 80)
axes[0].set_ylim(180, 40)
axes[0].set_title("Top map margin — longitude ticks")
axes[0].set_xlabel("Original pixel X")
axes[0].set_ylabel("Original pixel Y")
axes[0].set_xticks(range(100, width, 100))
axes[0].grid(alpha=0.25)

# Left margin: latitude labels and tick marks
axes[1].imshow(official_map)
axes[1].set_xlim(40, 320)
axes[1].set_ylim(2250, 70)
axes[1].set_title("Left map margin — latitude ticks")
axes[1].set_xlabel("Original pixel X")
axes[1].set_ylabel("Original pixel Y")
axes[1].set_xticks(range(40, 321, 20))
axes[1].set_yticks(range(100, 2251, 100))
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

print(f"Original dimensions: {width} × {height} pixels")

## Step 2.42 — Identify the Map-Frame Calibration Points

**Objective:** Establish the pixel positions of the official map's inner border and its printed latitude/longitude ticks.

**Why:** We need reliable pixel-to-coordinate control points before converting the official Pamba basin boundary into a georeferenced polygon.

**Important:** Use the original `PAMBA.jpg`, not a screenshot of the diagnostic plots. Do not use the outer image edges as map coordinates; the map frame is inset from those edges.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

image_path = "PAMBA.jpg"
img = Image.open(image_path).convert("RGB")

print("Image dimensions (width, height):", img.size)

# Display the original map at its native aspect ratio.
fig, ax = plt.subplots(figsize=(16, 11))
ax.imshow(img)
ax.set_title("Step 2.42 — Original Pamba Basin Map")
ax.set_xlabel("Pixel X")
ax.set_ylabel("Pixel Y")
ax.set_xlim(0, img.width - 1)
ax.set_ylim(img.height - 1, 0)
ax.grid(False)
plt.tight_layout()
plt.show()

## Step 2.43 — Inspect the Coordinate Ticks at Higher Resolution

**Objective:** Enlarge the top and left map margins to inspect the printed longitude and latitude ticks.

**Why:** The image includes titles, a legend, and other elements outside the geographic map. We must identify the actual coordinate ticks rather than estimate their positions from the full-image preview.

**Output:** Two enlarged crops for examining the longitude and latitude labels. No coordinates or map boundaries are changed in this step.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Load the original map image.
img = Image.open("PAMBA.jpg").convert("RGB")
width, height = img.size

# Crop the top margin containing longitude ticks.
top_crop = img.crop((80, 40, width - 60, 180))

# Crop the left margin containing latitude ticks.
left_crop = img.crop((70, 50, 430, height - 50))

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

axes[0].imshow(top_crop)
axes[0].set_title("Top margin — longitude ticks")
axes[0].set_xlabel("Crop-relative pixel X")
axes[0].set_ylabel("Crop-relative pixel Y")

axes[1].imshow(left_crop)
axes[1].set_title("Left margin — latitude ticks")
axes[1].set_xlabel("Crop-relative pixel X")
axes[1].set_ylabel("Crop-relative pixel Y")

plt.tight_layout()
plt.show()

## Step 2.44 — Improve Coordinate-Tick Inspection

**Objective:** Display the longitude and latitude margins independently at a useful size.

**Method:**
- Crop the top margin containing longitude labels.
- Crop the left margin containing latitude labels and rotate it for easier visual inspection.
- Preserve the original image and its pixel coordinates.

**Expected result:** Two readable diagnostic plots to help locate coordinate ticks accurately.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("PAMBA.jpg").convert("RGB")
width, height = img.size

# Top margin: longitude labels and the upper map frame.
top_crop = img.crop((80, 35, width - 60, 145))

plt.figure(figsize=(18, 3))
plt.imshow(top_crop)
plt.title("Step 2.44A — Longitude Tick Labels")
plt.xlabel("Crop-relative pixel X")
plt.ylabel("Crop-relative pixel Y")
plt.tight_layout()
plt.show()

# Left margin: latitude labels and the left map frame.
left_crop = img.crop((65, 45, 145, height - 45))

# Rotate for easier visual inspection.
left_crop_rotated = left_crop.rotate(90, expand=True)

plt.figure(figsize=(5, 18))
plt.imshow(left_crop_rotated)
plt.title("Step 2.44B — Latitude Tick Labels (Rotated)")
plt.xlabel("Rotated crop pixel X")
plt.ylabel("Rotated crop pixel Y")
plt.tight_layout()
plt.show()

## Step 2.45 — Correct the latitude tick inspection layout

Re-display the latitude labels from the official Pamba basin map using a landscape-oriented figure.

The previous inspection rotated the left margin correctly, but used portrait figure dimensions, compressing the resulting image. This step corrects the display proportions so that the latitude tick labels can be inspected clearly before coordinate calibration.

In [ ]:
import matplotlib.pyplot as plt

# Crop the left map margin containing latitude labels.
left_crop = img.crop((65, 45, 145, height - 45))

# Rotate the crop so latitude labels can be inspected horizontally.
left_crop_rotated = left_crop.rotate(90, expand=True)

# Display using landscape dimensions to preserve readability.
plt.figure(figsize=(18, 5))
plt.imshow(left_crop_rotated, aspect="auto", interpolation="nearest")
plt.title("Step 2.45 — Latitude Tick Labels (Corrected Display)")
plt.xlabel("Rotated crop pixel X")
plt.ylabel("Rotated crop pixel Y")
plt.tight_layout()
plt.show()

## Step 2.46 — Enlarge and isolate latitude tick labels

The previous inspection revealed the latitude tick marks, but the coordinate labels remain difficult to read.

This step:
- Narrows the crop around the left map margin.
- Enlarges the cropped image for easier inspection.
- Preserves the original image for subsequent pixel-coordinate calibration.

We will use the visible latitude labels and their corresponding pixel positions to establish the map's vertical coordinate reference.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Isolate the left map margin.
latitude_crop = img.crop((65, 45, 120, height - 45))

# Rotate and enlarge for readability.
latitude_crop_rotated = latitude_crop.rotate(90, expand=True)

latitude_crop_enlarged = latitude_crop_rotated.resize(
    (
        latitude_crop_rotated.width * 2,
        latitude_crop_rotated.height * 2
    ),
    Image.Resampling.LANCZOS
)

plt.figure(figsize=(18, 7))
plt.imshow(latitude_crop_enlarged)
plt.title("Step 2.46 — Enlarged Latitude Tick Labels")
plt.xlabel("Enlarged rotated pixel X")
plt.ylabel("Enlarged rotated pixel Y")
plt.tight_layout()
plt.show()

## Step 2.47 — Correct latitude label orientation

The enlarged latitude labels are visible, but their text is upside down.

This step rotates the cropped margin in the opposite direction so the latitude values can be read normally. We will preserve the same crop boundaries to avoid changing the tick positions.

In [ ]:
# Rotate in the opposite direction to correct label orientation.
latitude_crop_rotated = latitude_crop.rotate(270, expand=True)

# Enlarge the corrected crop.
latitude_crop_enlarged = latitude_crop_rotated.resize(
    (
        latitude_crop_rotated.width * 2,
        latitude_crop_rotated.height * 2
    ),
    Image.Resampling.LANCZOS
)

plt.figure(figsize=(18, 7))
plt.imshow(latitude_crop_enlarged)
plt.title("Step 2.47 — Latitude Tick Labels (Corrected Orientation)")
plt.xlabel("Enlarged rotated pixel X")
plt.ylabel("Enlarged rotated pixel Y")
plt.tight_layout()
plt.show()

## Step 2.48 — Convert latitude tick positions to original-image coordinates

The latitude labels are now readable in the rotated and enlarged crop.

We will:
1. Record the nine latitude reference values.
2. Record their approximate positions in the enlarged rotated image.
3. Convert those positions back to the original PAMBA.jpg Y-pixel coordinates.
4. Display the resulting reference table.

These are initial visual estimates. We will refine the pixel positions before performing the final geographic transformation.

In [ ]:
import pandas as pd

# Latitude labels identified from the official map.
latitude_values = [
    9 + 5/60,
    9 + 10/60,
    9 + 15/60,
    9 + 20/60,
    9 + 25/60,
    9 + 30/60,
    9 + 35/60,
    9 + 40/60,
    9 + 45/60
]

latitude_labels = [
    "9°05'N", "9°10'N", "9°15'N",
    "9°20'N", "9°25'N", "9°30'N",
    "9°35'N", "9°40'N", "9°45'N"
]

# Approximate tick positions in the enlarged rotated crop.
rotated_x_positions = [
    275, 760, 1250, 1740, 2230,
    2720, 3210, 3700, 4190
]

# Convert enlarged rotated X positions to original image Y positions.
crop_top = 45
original_crop_height = height - 90
enlargement_factor = 2

original_y_positions = [
    crop_top + original_crop_height - 1 - (x / enlargement_factor)
    for x in rotated_x_positions
]

latitude_reference = pd.DataFrame({
    "Latitude label": latitude_labels,
    "Latitude decimal": latitude_values,
    "Rotated crop X": rotated_x_positions,
    "Original image Y": original_y_positions
})

print("Latitude reference coordinates:")
display(latitude_reference.round(2))

## Step 2.49 — Verify latitude reference positions on the original map

The latitude reference table has been created from the enlarged rotated crop.

This step overlays the estimated original-image Y positions on the full official Pamba basin map.

We will visually verify that:
- Each horizontal reference line aligns with its corresponding latitude tick.
- The latitude sequence increases from south to north.
- The reference positions are consistent with the original map frame.

These positions will only be accepted for calibration after visual verification.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(16, 11))

ax.imshow(img)

# Overlay estimated latitude reference positions.
for _, row in latitude_reference.iterrows():
    y = row["Original image Y"]

    ax.axhline(
        y=y,
        color="red",
        linestyle="--",
        linewidth=0.8,
        alpha=0.7
    )

    ax.text(
        150,
        y - 5,
        row["Latitude label"],
        color="red",
        fontsize=8,
        backgroundcolor="white"
    )

ax.set_title("Step 2.49 — Latitude Reference Alignment Check")
ax.set_xlabel("Original image X (pixels)")
ax.set_ylabel("Original image Y (pixels)")

ax.set_xlim(0, width)
ax.set_ylim(height, 0)

plt.tight_layout()
plt.show()

## Step 2.50 — Calculate latitude calibration accuracy

Use the nine identified latitude reference points to fit a linear relationship between geographic latitude and original image Y coordinates.

Calculate:
- Linear regression equation
- R² score
- Residual error for each reference point
- Root Mean Square Error (RMSE)

This checks whether the latitude reference points follow a sufficiently consistent linear relationship for georeferencing.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Latitude reference coordinates
latitude_data = pd.DataFrame({
    "Latitude_Label": [
        "9°05'N", "9°10'N", "9°15'N",
        "9°20'N", "9°25'N", "9°30'N",
        "9°35'N", "9°40'N", "9°45'N"
    ],
    "Latitude_Decimal": [
        9 + 5/60, 9 + 10/60, 9 + 15/60,
        9 + 20/60, 9 + 25/60, 9 + 30/60,
        9 + 35/60, 9 + 40/60, 9 + 45/60
    ],
    "Original_Image_Y": [
        2156.5, 1914.0, 1669.0,
        1424.0, 1179.0, 934.0,
        689.0, 444.0, 199.0
    ]
})

X = latitude_data[["Latitude_Decimal"]]
y = latitude_data["Original_Image_Y"]

# Fit linear calibration model
latitude_model = LinearRegression()
latitude_model.fit(X, y)

predicted_y = latitude_model.predict(X)
residuals = y - predicted_y

r2 = r2_score(y, predicted_y)
rmse = np.sqrt(mean_squared_error(y, predicted_y))

latitude_data["Predicted_Image_Y"] = predicted_y
latitude_data["Residual_Pixels"] = residuals

print("LATITUDE CALIBRATION")
print("-" * 55)
print(f"Equation: Image Y = {latitude_model.coef_[0]:.4f} × Latitude "
      f"+ {latitude_model.intercept_:.4f}")
print(f"R² score: {r2:.8f}")
print(f"RMSE: {rmse:.4f} pixels")
print(f"Maximum absolute residual: {np.abs(residuals).max():.4f} pixels")

print("\nREFERENCE POINT RESIDUALS")
display(latitude_data.round(4))

## Step 2.50 — Calculate latitude calibration accuracy

Use the nine identified latitude reference points to fit a linear relationship between geographic latitude and original image Y coordinates.

Calculate:
- Linear regression equation
- R² score
- Residual error for each reference point
- Root Mean Square Error (RMSE)

This checks whether the latitude reference points follow a sufficiently consistent linear relationship for georeferencing.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Latitude reference coordinates
latitude_data = pd.DataFrame({
    "Latitude_Label": [
        "9°05'N", "9°10'N", "9°15'N",
        "9°20'N", "9°25'N", "9°30'N",
        "9°35'N", "9°40'N", "9°45'N"
    ],
    "Latitude_Decimal": [
        9 + 5/60, 9 + 10/60, 9 + 15/60,
        9 + 20/60, 9 + 25/60, 9 + 30/60,
        9 + 35/60, 9 + 40/60, 9 + 45/60
    ],
    "Original_Image_Y": [
        2156.5, 1914.0, 1669.0,
        1424.0, 1179.0, 934.0,
        689.0, 444.0, 199.0
    ]
})

X = latitude_data[["Latitude_Decimal"]]
y = latitude_data["Original_Image_Y"]

# Fit linear calibration model
latitude_model = LinearRegression()
latitude_model.fit(X, y)

predicted_y = latitude_model.predict(X)
residuals = y - predicted_y

r2 = r2_score(y, predicted_y)
rmse = np.sqrt(mean_squared_error(y, predicted_y))

latitude_data["Predicted_Image_Y"] = predicted_y
latitude_data["Residual_Pixels"] = residuals

print("LATITUDE CALIBRATION")
print("-" * 55)
print(f"Equation: Image Y = {latitude_model.coef_[0]:.4f} × Latitude "
      f"+ {latitude_model.intercept_:.4f}")
print(f"R² score: {r2:.8f}")
print(f"RMSE: {rmse:.4f} pixels")
print(f"Maximum absolute residual: {np.abs(residuals).max():.4f} pixels")

print("\nREFERENCE POINT RESIDUALS")
display(latitude_data.round(4))

## Step 2.51 — Inspect longitude reference positions

Extract the upper map margin from the original PAMBA.jpg image.

Display the longitude labels and their corresponding map-frame tick marks using original image pixel coordinates.

The objective is to identify the exact X coordinate of each longitude tick before fitting the longitude calibration model.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Load the original map
img = Image.open("PAMBA.jpg").convert("RGB")
width, height = img.size

# Extract the upper map margin
top_crop = img.crop((80, 35, width - 60, 145))

plt.figure(figsize=(20, 5))

plt.imshow(
    top_crop,
    extent=(80, width - 60, 145, 35),
    aspect="auto"
)

plt.title("Step 2.51 — Longitude Reference Positions")
plt.xlabel("Original Image X (pixels)")
plt.ylabel("Original Image Y (pixels)")

plt.xticks(range(0, width + 1, 100))
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Original image dimensions:", width, "×", height)
print("Longitude labels to identify:")
print("76°20'E through 77°15'E at 5-minute intervals")

## Step 2.52 — Enlarge longitude reference sections

Divide the upper margin of PAMBA.jpg into three overlapping sections.

Enlarge each section independently to make the longitude labels and their corresponding tick marks readable.

Preserve the original image pixel coordinates on the horizontal axis. These coordinates will be used to construct the longitude calibration reference table.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("PAMBA.jpg").convert("RGB")
width, height = img.size

# Original image coordinate ranges for three overlapping sections
sections = [
    (90, 1200, "Section 1 — Western longitude ticks"),
    (1100, 2200, "Section 2 — Central longitude ticks"),
    (2100, width - 60, "Section 3 — Eastern longitude ticks")
]

for x_start, x_end, title in sections:

    # Crop the upper map margin
    crop = img.crop((x_start, 35, x_end, 110))

    # Enlarge the crop for readability
    scale = 3
    enlarged = crop.resize(
        (crop.width * scale, crop.height * scale),
        Image.Resampling.LANCZOS
    )

    plt.figure(figsize=(18, 4))
    plt.imshow(
        enlarged,
        extent=(x_start, x_end, 110, 35),
        aspect="auto"
    )

    plt.title(f"Step 2.52 — {title}")
    plt.xlabel("Original Image X (pixels)")
    plt.ylabel("Original Image Y (pixels)")
    plt.xticks(range(x_start, x_end + 1, 50))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Step 2.53 — Record longitude reference coordinates

The longitude tick marks were inspected in three sections of the original Pamba basin map.

Each tick represents a 5-arcminute interval. The approximate horizontal pixel positions are recorded below and will be used to fit a linear pixel-to-longitude transformation.

These are initial visual measurements; calibration residuals will help identify whether any positions need refinement.

In [ ]:
import numpy as np
import pandas as pd

longitude_reference = pd.DataFrame({
    "Longitude_Label": [
        "76°20'E", "76°25'E", "76°30'E", "76°35'E",
        "76°40'E", "76°45'E", "76°50'E", "76°55'E",
        "77°00'E", "77°05'E", "77°10'E", "77°15'E"
    ],
    "Longitude_Decimal": [
        76 + 20/60, 76 + 25/60, 76 + 30/60, 76 + 35/60,
        76 + 40/60, 76 + 45/60, 76 + 50/60, 76 + 55/60,
        77.0, 77 + 5/60, 77 + 10/60, 77 + 15/60
    ],
    "Original_Image_X": [
        240, 490, 740, 990,
        1250, 1500, 1770, 2020,
        2270, 2520, 2775, 3025
    ]
})

display(longitude_reference)

## Step 2.54 — Calculate longitude calibration

Fit a linear regression between the original image's horizontal pixel coordinates (X) and longitude in decimal degrees.

Evaluate the calibration using:
- Regression equation
- R² score
- Root Mean Square Error (RMSE)
- Maximum absolute residual
- Individual reference-point residuals

The fitted equation will provide the pixel-to-longitude transformation needed to georeference the Pamba basin map.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd

# Prepare calibration variables
X_lon = longitude_reference[["Original_Image_X"]].values
y_lon = longitude_reference["Longitude_Decimal"].values

# Fit linear regression
longitude_model = LinearRegression()
longitude_model.fit(X_lon, y_lon)

# Predict longitude values
predicted_longitude = longitude_model.predict(X_lon)

# Calculate calibration metrics
longitude_r2 = r2_score(y_lon, predicted_longitude)
longitude_rmse = np.sqrt(
    mean_squared_error(y_lon, predicted_longitude)
)
longitude_max_residual = np.max(
    np.abs(y_lon - predicted_longitude)
)

# Store calibration parameters
longitude_slope = longitude_model.coef_[0]
longitude_intercept = longitude_model.intercept_

print("LONGITUDE CALIBRATION")
print("-" * 55)
print(
    f"Equation: Longitude = {longitude_slope:.8f} × "
    f"Image X + {longitude_intercept:.8f}"
)
print(f"R² score: {longitude_r2:.8f}")
print(f"RMSE: {longitude_rmse:.6f} degrees")
print(
    f"Maximum absolute residual: "
    f"{longitude_max_residual:.6f} degrees"
)

# Build residual table
longitude_residuals = longitude_reference.copy()
longitude_residuals["Predicted_Longitude"] = predicted_longitude
longitude_residuals["Residual_Degrees"] = (
    y_lon - predicted_longitude
)

# Convert residuals to approximate ground distance
longitude_residuals["Residual_Metres_Approx"] = (
    longitude_residuals["Residual_Degrees"]
    * 111320
    * np.cos(np.deg2rad(9.4))
)

print("\nREFERENCE POINT RESIDUALS")
display(longitude_residuals.round(7))

## Step 2.55 — Inspect longitude calibration residuals

Visualise the longitude calibration residuals against their original horizontal pixel positions.

The plot will help identify:
- Systematic positional bias.
- Local deviations from the fitted linear relationship.
- Potentially inaccurate pixel measurements.
- Whether a linear transformation is sufficiently appropriate.

The calibration will remain provisional until the residual pattern is assessed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Residuals in decimal degrees
axes[0].scatter(
    longitude_residuals["Original_Image_X"],
    longitude_residuals["Residual_Degrees"],
    s=55
)
axes[0].axhline(0, linestyle="--")
axes[0].set_title("Longitude Calibration Residuals")
axes[0].set_xlabel("Original Image X (pixels)")
axes[0].set_ylabel("Residual (degrees)")
axes[0].grid(alpha=0.3)

# Approximate ground-distance residuals
axes[1].scatter(
    longitude_residuals["Original_Image_X"],
    longitude_residuals["Residual_Metres_Approx"],
    s=55
)
axes[1].axhline(0, linestyle="--")
axes[1].set_title("Approximate Ground-Distance Residuals")
axes[1].set_xlabel("Original Image X (pixels)")
axes[1].set_ylabel("Residual (metres)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2.53 — Visualize Longitude Calibration Residuals

The longitude calibration model has been fitted using the geographic reference points extracted from the official Pamba River Basin map.

This step examines the residuals (actual longitude − predicted longitude) to understand how the calibration error varies across the image.

Two plots will be generated:

1. **Longitude Calibration Residuals:** Shows the difference between the reference longitude and the longitude predicted by the calibration equation.
2. **Approximate Ground-Distance Residuals:** Converts the longitude residuals into approximate ground distances in metres, using the study area's mean latitude.

The zero-reference line represents perfect agreement. Residuals distributed around this line indicate the direction and magnitude of calibration errors.

These plots help assess the consistency of the longitude calibration before proceeding with map georeferencing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Longitude reference coordinates
longitude_image_x = np.array([
    240, 490, 740, 990,
    1250, 1500, 1770, 2020,
    2270, 2520, 2775, 3025
], dtype=float)

longitude_reference = np.array([
    76.333333, 76.416667, 76.500000, 76.583333,
    76.666667, 76.750000, 76.833333, 76.916667,
    77.000000, 77.083333, 77.166667, 77.250000
], dtype=float)

# Longitude calibration equation
predicted_longitude = (
    0.00032809 * longitude_image_x + 76.25605765
)

# Residual = Reference longitude - Predicted longitude
longitude_residuals = (
    longitude_reference - predicted_longitude
)

# Convert longitude residuals into approximate ground distances.
# Mean latitude of the mapped region is approximately 9.4° N.
mean_latitude = 9.4

metres_per_degree_longitude = (
    111320 * np.cos(np.radians(mean_latitude))
)

distance_residuals = (
    longitude_residuals * metres_per_degree_longitude
)

# Plot calibration residuals
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].scatter(
    longitude_image_x,
    longitude_residuals,
    s=55
)

axes[0].axhline(
    y=0,
    linestyle="--",
    linewidth=1.5
)

axes[0].set_title("Longitude Calibration Residuals")
axes[0].set_xlabel("Original Image X (pixels)")
axes[0].set_ylabel("Residual (degrees)")
axes[0].grid(True, alpha=0.3)


# Plot approximate ground-distance residuals
axes[1].scatter(
    longitude_image_x,
    distance_residuals,
    s=55
)

axes[1].axhline(
    y=0,
    linestyle="--",
    linewidth=1.5
)

axes[1].set_title("Approximate Ground-Distance Residuals")
axes[1].set_xlabel("Original Image X (pixels)")
axes[1].set_ylabel("Residual (metres)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2.54 — Combine Latitude and Longitude Calibration

In this step, we combine the independently calculated latitude and longitude calibration equations into a single pixel-to-geographic-coordinate transformation.

Our calibrated relationships are:

**Longitude:**

Longitude = 0.00032809 × Image X + 76.25605765

**Latitude:**

Image Y = -2938.0000 × Latitude + 28844.8889

Rearranging the latitude equation:

Latitude = (28844.8889 − Image Y) / 2938

Together, these equations allow us to estimate the geographic coordinates of any pixel in the original map.

We will:

1. Define the combined transformation.
2. Convert selected pixel positions into longitude and latitude.
3. Check the transformed reference points against their known geographic coordinates.
4. Calculate the approximate geographic extent of the original image.

At this stage, we are verifying the transformation mathematically. We will not modify or overwrite the original map image.

In [ ]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Calibration coefficients
# --------------------------------------------------

LONGITUDE_SLOPE = 0.00032809
LONGITUDE_INTERCEPT = 76.25605765

LATITUDE_PIXEL_SLOPE = -2938.0000
LATITUDE_PIXEL_INTERCEPT = 28844.8889


# --------------------------------------------------
# 2. Combined pixel-to-geographic transformation
# --------------------------------------------------

def pixel_to_geographic(image_x, image_y):
    """
    Convert original image pixel coordinates
    into approximate geographic coordinates.
    """

    longitude = (
        LONGITUDE_SLOPE * np.asarray(image_x)
        + LONGITUDE_INTERCEPT
    )

    latitude = (
        LATITUDE_PIXEL_INTERCEPT - np.asarray(image_y)
    ) / abs(LATITUDE_PIXEL_SLOPE)

    return longitude, latitude


# --------------------------------------------------
# 3. Verify longitude reference points
# --------------------------------------------------

longitude_predicted, _ = pixel_to_geographic(
    longitude_image_x,
    np.zeros_like(longitude_image_x)
)

longitude_errors = longitude_reference - longitude_predicted


# --------------------------------------------------
# 4. Verify latitude reference points
# --------------------------------------------------

latitude_image_y = np.array([
    2156.5, 1914.0, 1669.0,
    1424.0, 1179.0, 934.0,
    689.0, 444.0, 199.0
])

latitude_reference = np.array([
    9.083333, 9.166667, 9.250000,
    9.333333, 9.416667, 9.500000,
    9.583333, 9.666667, 9.750000
])

_, latitude_predicted = pixel_to_geographic(
    np.zeros_like(latitude_image_y),
    latitude_image_y
)

latitude_errors = latitude_reference - latitude_predicted


# --------------------------------------------------
# 5. Display reference-point verification
# --------------------------------------------------

longitude_check = pd.DataFrame({
    "Reference_Longitude": longitude_reference,
    "Predicted_Longitude": longitude_predicted,
    "Residual_Degrees": longitude_errors
})

latitude_check = pd.DataFrame({
    "Reference_Latitude": latitude_reference,
    "Predicted_Latitude": latitude_predicted,
    "Residual_Degrees": latitude_errors
})

print("LONGITUDE TRANSFORMATION CHECK")
display(longitude_check.round(6))

print("\nLATITUDE TRANSFORMATION CHECK")
display(latitude_check.round(6))


# --------------------------------------------------
# 6. Calculate original image geographic extent
# --------------------------------------------------

from PIL import Image

img = Image.open("PAMBA.jpg")
image_width, image_height = img.size

corner_x = np.array([
    0,
    image_width,
    image_width,
    0
])

corner_y = np.array([
    0,
    0,
    image_height,
    image_height
])

corner_lon, corner_lat = pixel_to_geographic(
    corner_x,
    corner_y
)

extent = pd.DataFrame({
    "Corner": [
        "Top-left",
        "Top-right",
        "Bottom-right",
        "Bottom-left"
    ],
    "Longitude": corner_lon,
    "Latitude": corner_lat
})

print("\nAPPROXIMATE ORIGINAL IMAGE GEOGRAPHIC EXTENT")
display(extent.round(6))

print("\nImage dimensions:", image_width, "×", image_height)

## Step 2.55 — Identify the Original Map Frame

The previous step calculated geographic coordinates for the complete image. However, the actual Pamba Basin map is enclosed within an inner rectangular frame, while the surrounding margins contain coordinate labels, the title, legend, scale bar and other map elements.

For accurate georeferencing, we need to identify the pixel coordinates of the four inner map-frame edges.

In this step, we will:

1. Display the original image with pixel-coordinate axes.
2. Inspect the inner rectangular map frame.
3. Identify its approximate left, right, top and bottom pixel positions.

We will use the inner map frame rather than the outer image boundary for the next geographic extent calculation.

Do not crop, resize or modify the original image.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Load the original map
img = Image.open("PAMBA.jpg").convert("RGB")
image_array = np.array(img)

height, width = image_array.shape[:2]

print(f"Original image dimensions: {width} × {height} pixels")

# Display the complete image with pixel coordinates
plt.figure(figsize=(18, 13))

plt.imshow(image_array, origin="upper")

plt.title("Step 2.55 — Identify the Original Map Frame")
plt.xlabel("Original Image X (pixels)")
plt.ylabel("Original Image Y (pixels)")

plt.xticks(np.arange(0, width + 1, 200))
plt.yticks(np.arange(0, height + 1, 200))

plt.grid(True, alpha=0.25)
plt.xlim(0, width)
plt.ylim(height, 0)

plt.tight_layout()
plt.show()

## Step 2.56 — Mark and Verify the Inner Map Frame

Based on visual inspection, we have identified approximate pixel coordinates for the four edges of the inner map frame:

- Left: X = 107
- Right: X = 3221
- Top: Y = 64
- Bottom: Y = 2264

This step overlays these coordinates on the original image to verify that they follow the rectangular map border.

We will also display the extracted map-frame region separately.

These coordinates are provisional. We will adjust them if the overlay reveals any misalignment.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Load original map
img = Image.open("PAMBA.jpg").convert("RGB")

width, height = img.size

# Approximate inner map-frame coordinates
FRAME_LEFT = 107
FRAME_RIGHT = 3221
FRAME_TOP = 64
FRAME_BOTTOM = 2264

# Validate coordinates
assert 0 <= FRAME_LEFT < FRAME_RIGHT <= width
assert 0 <= FRAME_TOP < FRAME_BOTTOM <= height

# Display original image with frame boundaries
fig, ax = plt.subplots(figsize=(18, 13))

ax.imshow(img)

ax.axvline(FRAME_LEFT, color="red", linestyle="--", linewidth=1.5)
ax.axvline(FRAME_RIGHT, color="red", linestyle="--", linewidth=1.5)
ax.axhline(FRAME_TOP, color="red", linestyle="--", linewidth=1.5)
ax.axhline(FRAME_BOTTOM, color="red", linestyle="--", linewidth=1.5)

ax.set_title("Step 2.56A — Inner Map Frame Alignment")
ax.set_xlabel("Original Image X (pixels)")
ax.set_ylabel("Original Image Y (pixels)")

ax.grid(False)
plt.tight_layout()
plt.show()

# Extract the approximate map-frame region
map_frame_crop = img.crop((
    FRAME_LEFT,
    FRAME_TOP,
    FRAME_RIGHT,
    FRAME_BOTTOM
))

plt.figure(figsize=(16, 11))
plt.imshow(map_frame_crop)
plt.title("Step 2.56B — Extracted Inner Map Frame")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Original image dimensions:", width, "×", height)
print("Provisional map-frame boundaries:")
print("Left:", FRAME_LEFT)
print("Right:", FRAME_RIGHT)
print("Top:", FRAME_TOP)
print("Bottom:", FRAME_BOTTOM)
print(
    "Map-frame dimensions:",
    FRAME_RIGHT - FRAME_LEFT,
    "×",
    FRAME_BOTTOM - FRAME_TOP
)

## Step 2.57 — Check the Calibrated Map-Frame Corners

We have identified provisional pixel coordinates for the inner map frame and calibrated the image's pixel coordinates to longitude and latitude.

This step calculates the geographic coordinates of the four frame corners using those calibration equations. We will compare the calculated extent with the printed map coordinates as a consistency check before proceeding.

This does not create a new calibration. If the calculated corners differ noticeably from the map's printed extent, we should revisit the frame edges or calibration points.

Provisional inner-frame bounds:

- Left: X = 107
- Right: X = 3221
- Top: Y = 64
- Bottom: Y = 2264

In [ ]:
# Step 2.57 — Calculate the calibrated geographic coordinates
# of the four provisional inner-map-frame corners.

# Calibration equations from the previous steps:
# Longitude = 0.00032809 * Image_X + 76.25605765
# Latitude  = -0.00034037 * Image_Y + 9.817866
#
# If your latest calibration equations differ, replace these
# coefficients with the exact values from your calibration output.

longitude_slope = 0.00032809
longitude_intercept = 76.25605765

latitude_slope = -0.00034037
latitude_intercept = 9.817866

# Provisional inner map-frame bounds in original-image pixels.
left_x = 107
right_x = 3221
top_y = 64
bottom_y = 2264

def pixel_to_longitude(x):
    return longitude_slope * x + longitude_intercept

def pixel_to_latitude(y):
    return latitude_slope * y + latitude_intercept

corners = {
    "Top-left": (left_x, top_y),
    "Top-right": (right_x, top_y),
    "Bottom-right": (right_x, bottom_y),
    "Bottom-left": (left_x, bottom_y),
}

print("CALIBRATED INNER MAP-FRAME CORNERS")
print("-" * 60)

for corner_name, (x, y) in corners.items():
    longitude = pixel_to_longitude(x)
    latitude = pixel_to_latitude(y)

    print(
        f"{corner_name:13s} | "
        f"X={x:4d}, Y={y:4d} | "
        f"Longitude={longitude:.6f}°E, "
        f"Latitude={latitude:.6f}°N"
    )

print("\nCALCULATED MAP EXTENT")
print("-" * 60)
print(
    f"Longitude: {pixel_to_longitude(left_x):.6f}°E "
    f"to {pixel_to_longitude(right_x):.6f}°E"
)
print(
    f"Latitude:  {pixel_to_latitude(bottom_y):.6f}°N "
    f"to {pixel_to_latitude(top_y):.6f}°N"
)

## Step 2.58 — Compare the Calculated Extent with the Printed Map Ticks

The calibrated inner-frame corners give an approximate extent of:

- Longitude: 76.291163°E to 77.312836°E
- Latitude: 9.047268°N to 9.796082°N

The printed map ticks indicate a wider coordinate range. This step compares the calculated frame extent with the nearest printed tick labels on each side.

The comparison is a diagnostic check. A difference may come from the provisional frame-edge pixels, the approximate calibration equations, or the fact that the printed ticks are not located exactly on the frame corners.

Do not use the calculated extent as a final georeferencing result until this comparison is reviewed.

In [ ]:
# Step 2.58 — Compare the calculated frame extent
# with the printed coordinate ticks.

# Calculated extent from Step 2.57.
calculated_west = 76.291163
calculated_east = 77.312836
calculated_south = 9.047268
calculated_north = 9.796082

# Approximate printed tick coverage visible on the map.
# Replace these with the exact outermost tick values if needed.
printed_west = 76 + 20 / 60
printed_east = 77 + 15 / 60
printed_south = 9 + 5 / 60
printed_north = 9 + 45 / 60

comparisons = [
    ("West", calculated_west, printed_west),
    ("East", calculated_east, printed_east),
    ("South", calculated_south, printed_south),
    ("North", calculated_north, printed_north),
]

print("FRAME EXTENT VS. PRINTED TICKS")
print("-" * 72)

for side, calculated, printed in comparisons:
    difference = calculated - printed
    print(
        f"{side:6s} | "
        f"Calculated={calculated:.6f}° | "
        f"Printed tick={printed:.6f}° | "
        f"Difference={difference:+.6f}°"
    )

## Step 2.59 — Validate Calibration at the Printed Coordinate Ticks

Step 2.58 compared the map-frame edges with the outermost printed ticks. That was not a like-for-like comparison: the frame extends beyond the tick marks.

This step checks the calibration correctly by:
1. Using each printed longitude and latitude value as a reference.
2. Predicting its image-pixel position from the calibration equations.
3. Comparing the predicted position with the observed tick position.
4. Reporting pixel residuals and summary errors.

A small residual means the calibration predicts the observed tick location well. This check evaluates the calibration against the tick marks; it does not independently verify the provisional frame edges.

In [ ]:
# Step 2.59 — Validate the pixel-to-coordinate calibration
# by predicting the image-pixel positions of the printed ticks.

import pandas as pd
import numpy as np

# Use the calibration equations from the earlier steps.
# Longitude = longitude_slope * Image_X + longitude_intercept
# Latitude  = latitude_slope * Image_Y + latitude_intercept

longitude_slope = 0.00032809
longitude_intercept = 76.25605765

latitude_slope = -0.00034037
latitude_intercept = 9.817866

# Observed longitude tick positions from Step 2.53.
longitude_ticks = pd.DataFrame({
    "Label": [
        "76°20'E", "76°25'E", "76°30'E", "76°35'E",
        "76°40'E", "76°45'E", "76°50'E", "76°55'E",
        "77°00'E", "77°05'E", "77°10'E", "77°15'E"
    ],
    "Coordinate": [
        76 + 20/60, 76 + 25/60, 76 + 30/60, 76 + 35/60,
        76 + 40/60, 76 + 45/60, 76 + 50/60, 76 + 55/60,
        77.0, 77 + 5/60, 77 + 10/60, 77 + 15/60
    ],
    "Observed_Pixel": [
        240, 490, 740, 990, 1250, 1500,
        1770, 2020, 2270, 2520, 2775, 3025
    ]
})

# Observed latitude tick positions from Step 2.48.
latitude_ticks = pd.DataFrame({
    "Label": [
        "9°05'N", "9°10'N", "9°15'N", "9°20'N", "9°25'N",
        "9°30'N", "9°35'N", "9°40'N", "9°45'N"
    ],
    "Coordinate": [
        9 + 5/60, 9 + 10/60, 9 + 15/60, 9 + 20/60,
        9 + 25/60, 9 + 30/60, 9 + 35/60, 9 + 40/60,
        9 + 45/60
    ],
    "Observed_Pixel": [
        2156.5, 1914.0, 1669.0, 1424.0, 1179.0,
        934.0, 689.0, 444.0, 199.0
    ]
})

# Invert the calibration equations to predict pixel positions.
longitude_ticks["Predicted_Pixel"] = (
    longitude_ticks["Coordinate"] - longitude_intercept
) / longitude_slope

latitude_ticks["Predicted_Pixel"] = (
    latitude_ticks["Coordinate"] - latitude_intercept
) / latitude_slope

# Residual convention: observed pixel minus predicted pixel.
longitude_ticks["Residual_Pixels"] = (
    longitude_ticks["Observed_Pixel"] - longitude_ticks["Predicted_Pixel"]
)

latitude_ticks["Residual_Pixels"] = (
    latitude_ticks["Observed_Pixel"] - latitude_ticks["Predicted_Pixel"]
)

print("LONGITUDE TICK-POSITION CHECK")
display(longitude_ticks.round(3))

print("\nLATITUDE TICK-POSITION CHECK")
display(latitude_ticks.round(3))

print("\nSUMMARY")
print("-" * 55)

for axis_name, table in [
    ("Longitude", longitude_ticks),
    ("Latitude", latitude_ticks)
]:
    residuals = table["Residual_Pixels"].to_numpy()

    print(
        f"{axis_name}: "
        f"mean absolute residual = {np.mean(np.abs(residuals)):.3f} px; "
        f"maximum absolute residual = {np.max(np.abs(residuals)):.3f} px"
    )

## Step 2.60 — Check the Calibrated Frame Against Printed Coordinate Ticks

This step checks whether the calibrated inner map-frame extent agrees with the longitude and latitude values printed around the map.

The frame-edge coordinates do not need to match the nearest printed tick exactly: the map frame extends beyond the outermost labeled ticks. Instead, we compare the pixel positions of the printed ticks with the positions predicted by the calibration equations.

The longitude check has a maximum residual of 10.496 pixels and a mean absolute residual of 4.080 pixels. The latitude check has a maximum residual of 1.542 pixels and a mean absolute residual of 0.497 pixels.

The latitude alignment is strong. Longitude residuals are larger and show some variation, so the longitude calibration should be treated as provisional and checked again during georeferencing. These residuals measure alignment with the tick positions—not the overall geographic accuracy of the map.

In [ ]:
# Step 2.57 — Check the Calibrated Frame Against Printed Coordinate Ticks

import pandas as pd

longitude_tick_check = pd.DataFrame({
    "Label": [
        "76°20'E", "76°25'E", "76°30'E", "76°35'E",
        "76°40'E", "76°45'E", "76°50'E", "76°55'E",
        "77°00'E", "77°05'E", "77°10'E", "77°15'E"
    ],
    "Coordinate": [
        76.333333, 76.416667, 76.500000, 76.583333,
        76.666667, 76.750000, 76.833333, 76.916667,
        77.000000, 77.083333, 77.166667, 77.250000
    ],
    "Observed_Pixel": [
        240, 490, 740, 990, 1250, 1500,
        1770, 2020, 2270, 2520, 2775, 3025
    ],
    "Predicted_Pixel": [
        235.532, 489.527, 743.523, 997.518,
        1251.513, 1505.509, 1759.504, 2013.499,
        2267.495, 2521.490, 2775.485, 3029.481
    ],
    "Residual_Pixels": [
        4.468, 0.473, -3.523, -7.518, -1.513, -5.509,
        10.496, 6.501, 2.505, -1.490, -0.485, -4.481
    ]
})

latitude_tick_check = pd.DataFrame({
    "Label": [
        "9°05'N", "9°10'N", "9°15'N", "9°20'N", "9°25'N",
        "9°30'N", "9°35'N", "9°40'N", "9°45'N"
    ],
    "Coordinate": [
        9.083333, 9.166667, 9.250000, 9.333333, 9.416667,
        9.500000, 9.583333, 9.666667, 9.750000
    ],
    "Observed_Pixel": [
        2156.5, 1914.0, 1669.0, 1424.0, 1179.0,
        934.0, 689.0, 444.0, 199.0
    ],
    "Predicted_Pixel": [
        2158.042, 1913.210, 1668.379, 1423.547, 1178.715,
        933.884, 689.052, 444.221, 199.389
    ],
    "Residual_Pixels": [
        -1.542, 0.790, 0.621, 0.453, 0.285,
        0.116, -0.052, -0.221, -0.389
    ]
})

print("LONGITUDE TICK-POSITION CHECK")
display(longitude_tick_check)

print("\nLATITUDE TICK-POSITION CHECK")
display(latitude_tick_check)

longitude_mae = longitude_tick_check["Residual_Pixels"].abs().mean()
longitude_max = longitude_tick_check["Residual_Pixels"].abs().max()

latitude_mae = latitude_tick_check["Residual_Pixels"].abs().mean()
latitude_max = latitude_tick_check["Residual_Pixels"].abs().max()

print("\nSUMMARY")
print(
    f"Longitude: mean absolute residual = {longitude_mae:.3f} px; "
    f"maximum absolute residual = {longitude_max:.3f} px"
)
print(
    f"Latitude: mean absolute residual = {latitude_mae:.3f} px; "
    f"maximum absolute residual = {latitude_max:.3f} px"
)

## Step 2.61 — Refine the Longitude Calibration

The Step 2.60 check showed that latitude tick positions align closely, while longitude tick positions have larger residuals. The longitude residuals also change direction across the image, so we should refine the horizontal calibration before using it for georeferencing.

This step fits a new straight-line relationship between the observed longitude-tick pixel positions and their printed longitude values:

Longitude = slope × Image_X + intercept

We will report the updated equation and its residuals. This is still a calibration against the map's printed ticks, not an independent accuracy assessment. The provisional map-frame bounds remain unchanged.

In [ ]:
# Step 2.61 — Refine the longitude calibration from observed tick positions.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Printed longitude ticks and their observed positions
# in the original 3306 × 2340 image.
longitude_ticks = pd.DataFrame({
    "Label": [
        "76°20'E", "76°25'E", "76°30'E", "76°35'E",
        "76°40'E", "76°45'E", "76°50'E", "76°55'E",
        "77°00'E", "77°05'E", "77°10'E", "77°15'E"
    ],
    "Longitude": [
        76 + 20/60, 76 + 25/60, 76 + 30/60, 76 + 35/60,
        76 + 40/60, 76 + 45/60, 76 + 50/60, 76 + 55/60,
        77.0, 77 + 5/60, 77 + 10/60, 77 + 15/60
    ],
    "Image_X": [
        240, 490, 740, 990, 1250, 1500,
        1770, 2020, 2270, 2520, 2775, 3025
    ]
})

# Fit longitude as a function of original-image X.
longitude_slope, longitude_intercept = np.polyfit(
    longitude_ticks["Image_X"],
    longitude_ticks["Longitude"],
    1
)

longitude_ticks["Predicted_Longitude"] = (
    longitude_slope * longitude_ticks["Image_X"]
    + longitude_intercept
)

longitude_ticks["Residual_Degrees"] = (
    longitude_ticks["Longitude"]
    - longitude_ticks["Predicted_Longitude"]
)

longitude_ticks["Residual_Metres_Approx"] = (
    longitude_ticks["Residual_Degrees"]
    * 111_320
    * np.cos(np.deg2rad(9.4))
)

# Summary statistics
residuals = longitude_ticks["Residual_Degrees"].to_numpy()
ss_res = np.sum(residuals ** 2)
ss_tot = np.sum(
    (longitude_ticks["Longitude"] - longitude_ticks["Longitude"].mean()) ** 2
)
r_squared = 1 - ss_res / ss_tot
rmse_degrees = np.sqrt(np.mean(residuals ** 2))

print("REFINED LONGITUDE CALIBRATION")
print("-" * 60)
print(
    f"Longitude = {longitude_slope:.12f} × Image_X "
    f"+ {longitude_intercept:.12f}"
)
print(f"R²: {r_squared:.8f}")
print(f"RMSE: {rmse_degrees:.8f} degrees")
print(
    "Maximum absolute residual:",
    f"{np.max(np.abs(residuals)):.8f} degrees"
)
print(
    "Maximum absolute residual, approximate:",
    f"{np.max(np.abs(longitude_ticks['Residual_Metres_Approx'])):.1f} m"
)

print("\nREFERENCE-TICK RESIDUALS")
display(longitude_ticks.round(6))

# Plot observed ticks and the fitted calibration line.
x_line = np.linspace(
    longitude_ticks["Image_X"].min(),
    longitude_ticks["Image_X"].max(),
    200
)
y_line = longitude_slope * x_line + longitude_intercept

plt.figure(figsize=(10, 5))
plt.scatter(
    longitude_ticks["Image_X"],
    longitude_ticks["Longitude"],
    label="Observed longitude ticks"
)
plt.plot(x_line, y_line, label="Refined linear fit")
plt.xlabel("Original image X (pixels)")
plt.ylabel("Longitude (degrees east)")
plt.title("Step 2.61 — Refined Longitude Calibration")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Step 2.62 — Check the Combined Map Grid

Before georeferencing, compare the longitude and latitude tick residuals from their fitted straight-line models. This checks whether the printed grid is broadly consistent with a simple rectangular pixel-to-coordinate relationship.

The tick locations were read from the image, so residuals also include pixel-reading uncertainty. This is a diagnostic check, not proof of georeferencing accuracy. Keep the map-frame bounds provisional.

In [ ]:
# Step 2.62 — Check longitude and latitude calibration residuals together.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Longitude tick references from Step 2.61.
lon = np.array([
    76 + 20/60, 76 + 25/60, 76 + 30/60, 76 + 35/60,
    76 + 40/60, 76 + 45/60, 76 + 50/60, 76 + 55/60,
    77.0, 77 + 5/60, 77 + 10/60, 77 + 15/60
])
x_px = np.array([
    240, 490, 740, 990, 1250, 1500,
    1770, 2020, 2270, 2520, 2775, 3025
])

# Latitude tick references used in the earlier calibration.
lat = np.array([
    9 + 5/60, 9 + 10/60, 9 + 15/60, 9 + 20/60,
    9 + 25/60, 9 + 30/60, 9 + 35/60, 9 + 40/60, 9 + 45/60
])
y_px = np.array([2156.5, 1914, 1669, 1424, 1179, 934, 689, 444, 199])

# Fit coordinate values directly against original-image pixel positions.
lon_slope, lon_intercept = np.polyfit(x_px, lon, 1)
lat_slope, lat_intercept = np.polyfit(y_px, lat, 1)

lon_residual = lon - (lon_slope * x_px + lon_intercept)
lat_residual = lat - (lat_slope * y_px + lat_intercept)

# Approximate ground-distance equivalents near 9.4°N.
lon_residual_m = lon_residual * 111_320 * np.cos(np.deg2rad(9.4))
lat_residual_m = lat_residual * 111_320

summary = pd.DataFrame({
    "Axis": ["Longitude", "Latitude"],
    "Tick_Count": [len(lon_residual), len(lat_residual)],
    "RMSE_Degrees": [
        np.sqrt(np.mean(lon_residual**2)),
        np.sqrt(np.mean(lat_residual**2))
    ],
    "Max_Abs_Residual_Degrees": [
        np.max(np.abs(lon_residual)),
        np.max(np.abs(lat_residual))
    ],
    "Max_Abs_Residual_Metres_Approx": [
        np.max(np.abs(lon_residual_m)),
        np.max(np.abs(lat_residual_m))
    ]
})

print("COMBINED GRID CHECK")
display(summary.round(6))

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

axes[0].axhline(0, color="black", linewidth=1)
axes[0].scatter(x_px, lon_residual_m)
axes[0].plot(x_px, lon_residual_m)
axes[0].set_title("Longitude tick residuals")
axes[0].set_xlabel("Original image X (pixels)")
axes[0].set_ylabel("Approx. residual (m)")
axes[0].grid(True, alpha=0.3)

axes[1].axhline(0, color="black", linewidth=1)
axes[1].scatter(y_px, lat_residual_m)
axes[1].plot(y_px, lat_residual_m)
axes[1].set_title("Latitude tick residuals")
axes[1].set_xlabel("Original image Y (pixels)")
axes[1].set_ylabel("Approx. residual (m)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2.63 — Inspect the Printed Grid and Map Frame

Before refining the calibration further, visually inspect the official Pamba basin map at full resolution. Check whether the graticule lines are straight and parallel, whether they meet the frame consistently, and whether the tick positions used in the calibration were measured at the same point on each tick.

This is a visual quality-control step only. Do not change the calibration or georeference the map yet.

In [ ]:
# Step 2.63 — Inspect the original map, frame, and printed graticule.

from PIL import Image
import matplotlib.pyplot as plt

# Use the original downloaded map image.
map_image = Image.open("PAMBA.jpg").convert("RGB")

print("Image dimensions:", map_image.size)

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(map_image)
ax.set_title("Step 2.63 — Original Pamba Basin Map")
ax.set_xlabel("Original image X (pixels)")
ax.set_ylabel("Original image Y (pixels)")

# Add a light pixel grid to help inspect line/frame alignment.
ax.set_xticks(range(0, map_image.width + 1, 250))
ax.set_yticks(range(0, map_image.height + 1, 250))
ax.grid(True, linewidth=0.5, alpha=0.35)

plt.tight_layout()
plt.show()

## Step 2.64 — Crop the Map Frame and Graticule

Create a close-up of the map frame, including the longitude ticks along the top and bottom and the latitude ticks along the left and right. This makes it easier to compare tick marks and frame intersections without the basin labels and legend distracting from the inspection.

The crop is for visual checking only. Keep the current calibration and frame coordinates unchanged.

In [ ]:
# Step 2.64 — Display close-ups of the map frame and printed graticule.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

# Approximate frame coordinates identified earlier.
left, right = 107, 3221
top, bottom = 64, 2264

# Crop a narrow band around each frame edge.
top_crop = map_image.crop((left - 30, top - 35, right + 30, top + 130))
bottom_crop = map_image.crop((left - 30, bottom - 130, right + 30, bottom + 35))
left_crop = map_image.crop((left - 35, top - 30, left + 150, bottom + 30))
right_crop = map_image.crop((right - 150, top - 30, right + 35, bottom + 30))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].imshow(top_crop)
axes[0, 0].set_title("Top frame — longitude ticks")
axes[0, 0].axis("off")

axes[0, 1].imshow(bottom_crop)
axes[0, 1].set_title("Bottom frame — longitude ticks")
axes[0, 1].axis("off")

axes[1, 0].imshow(left_crop)
axes[1, 0].set_title("Left frame — latitude ticks")
axes[1, 0].axis("off")

axes[1, 1].imshow(right_crop)
axes[1, 1].set_title("Right frame — latitude ticks")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## Step 2.65 — Enlarge the Frame Edges

Display the top, bottom, left, and right frame edges in separate enlarged views so the printed tick marks and labels are easier to inspect. Keep each crop's original pixel coordinates in its title for reference.

This is still visual inspection only. Do not update the calibration or georeference the map yet.

In [ ]:
# Step 2.65 — Enlarge each map-frame edge separately.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

# Approximate frame coordinates identified earlier.
left, right = 107, 3221
top, bottom = 64, 2264

# Crops include the frame and nearby tick labels.
edge_crops = {
    "Top": (left - 20, 0, right + 20, top + 90),
    "Bottom": (left - 20, bottom - 90, right + 20, map_image.height),
    "Left": (0, top - 20, left + 90, bottom + 20),
    "Right": (right - 90, top - 20, map_image.width, bottom + 20),
}

fig, axes = plt.subplots(4, 1, figsize=(18, 14))

for ax, (edge_name, crop_box) in zip(axes, edge_crops.items()):
    crop = map_image.crop(crop_box)

    # Enlarge narrow edge strips for easier visual inspection.
    scale = 2 if edge_name in ("Top", "Bottom") else 4
    enlarged = crop.resize(
        (crop.width * scale, crop.height * scale),
        Image.Resampling.LANCZOS
    )

    ax.imshow(enlarged)
    ax.set_title(f"{edge_name} edge — source crop {crop_box}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Step 2.66 — Inspect the Latitude Ticks

Rotate the left and right frame strips so their latitude tick labels are displayed horizontally. Compare the labels and tick marks on both sides, especially near the top and bottom of the frame.

This is a visual check only. Keep the calibration and frame coordinates unchanged.

In [ ]:
# Step 2.66 — Rotate the left and right frame strips for easier inspection.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

# Approximate frame coordinates identified earlier.
left, right = 107, 3221
top, bottom = 64, 2264

left_strip = map_image.crop((0, top - 20, left + 90, bottom + 20))
right_strip = map_image.crop((right - 90, top - 20, map_image.width, bottom + 20))

# Rotate 90 degrees so the vertical tick labels become horizontal.
left_rotated = left_strip.rotate(90, expand=True)
right_rotated = right_strip.rotate(90, expand=True)

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].imshow(left_rotated)
axes[0].set_title("Left frame — latitude ticks (rotated)")
axes[0].axis("off")

axes[1].imshow(right_rotated)
axes[1].set_title("Right frame — latitude ticks (rotated)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Step 2.67 — Locate Tick Marks in the Original Image

Use the original image coordinates to inspect the frame and nearby tick marks at higher magnification. The plotted crops from earlier steps were scaled and rotated for viewing, which makes pixel positions harder to read reliably.

This step displays focused, pixel-coordinate views of the four frame edges. Use the visible tick marks—not the label text—to refine the reference pixel positions. Do not update the calibration yet.

In [ ]:
# Step 2.67 — Inspect tick marks in original-image pixel coordinates.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

# Approximate frame coordinates from the earlier visual inspection.
left, right = 107, 3221
top, bottom = 64, 2264

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Top edge: longitude ticks
top_crop = map_image.crop((left - 20, top - 25, right + 20, top + 65))
axes[0, 0].imshow(top_crop)
axes[0, 0].set_title("Top frame — original pixel coordinates")
axes[0, 0].set_xticks(range(0, top_crop.width + 1, 250))
axes[0, 0].grid(True, alpha=0.35)

# Bottom edge: longitude ticks
bottom_crop = map_image.crop((left - 20, bottom - 65, right + 20, bottom + 25))
axes[0, 1].imshow(bottom_crop)
axes[0, 1].set_title("Bottom frame — original pixel coordinates")
axes[0, 1].set_xticks(range(0, bottom_crop.width + 1, 250))
axes[0, 1].grid(True, alpha=0.35)

# Left edge: latitude ticks
left_crop = map_image.crop((left - 30, top - 10, left + 65, bottom + 10))
axes[1, 0].imshow(left_crop)
axes[1, 0].set_title("Left frame — original pixel coordinates")
axes[1, 0].set_yticks(range(0, left_crop.height + 1, 250))
axes[1, 0].grid(True, alpha=0.35)

# Right edge: latitude ticks
right_crop = map_image.crop((right - 65, top - 10, right + 30, bottom + 10))
axes[1, 1].imshow(right_crop)
axes[1, 1].set_title("Right frame — original pixel coordinates")
axes[1, 1].set_yticks(range(0, right_crop.height + 1, 250))
axes[1, 1].grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

## Step 2.68 — Inspect the Top Longitude Ticks

Display only the top frame at a larger scale, keeping the original image pixel coordinates visible. Focus on the centers of the printed tick marks, not the text labels.

Use this view to verify the longitude reference positions before changing the calibration. This step is visual inspection only.

In [ ]:
# Step 2.68 — Enlarge the top frame while retaining original pixel coordinates.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

# Crop the top frame and longitude labels.
x_min, x_max = 80, 3250
y_min, y_max = 25, 115
top_crop = map_image.crop((x_min, y_min, x_max, y_max))

fig, ax = plt.subplots(figsize=(20, 5))

# Enlarge for viewing; axes are converted back to original-image pixels.
scale = 4
ax.imshow(
    top_crop.resize(
        (top_crop.width * scale, top_crop.height * scale),
        Image.Resampling.LANCZOS
    ),
    extent=(x_min, x_max, y_max, y_min),
    aspect="auto"
)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_max, y_min)
ax.set_xticks(range(100, 3251, 100))
ax.set_yticks(range(25, 116, 10))
ax.set_xlabel("Original image X (pixels)")
ax.set_ylabel("Original image Y (pixels)")
ax.set_title("Step 2.68 — Top Frame Longitude Tick Marks")
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

## Step 2.69 — Inspect Individual Longitude Ticks

Isolate a small section around one known longitude tick from the original map image. Display the crop at its native pixel resolution so the tick itself is not distorted by enlargement.

This is a visual check only. Do not change the calibration or map-frame coordinates yet.

In [ ]:
# Step 2.69 — Inspect one longitude tick at native resolution.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

# Approximate original-image position of the 76°30'E top tick.
# Adjust only the crop window if the tick is not centered.
tick_x = 740
crop_box = (
    tick_x - 45,
    35,
    tick_x + 45,
    115
)

tick_crop = map_image.crop(crop_box)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(tick_crop, interpolation="nearest")
ax.set_title(f"Top longitude tick near X={tick_x}px — native pixels")
ax.set_xlabel("Crop X (pixels)")
ax.set_ylabel("Crop Y (pixels)")
ax.set_xticks(range(0, tick_crop.width + 1, 10))
ax.set_yticks(range(0, tick_crop.height + 1, 10))
ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.show()

## Step 2.70 — Inspect the Longitude Ticks Individually

Display each top-edge longitude tick in its own native-resolution crop. The crop centers use the earlier approximate X positions; the goal is to readjust them by locating the center of each vertical tick mark.

Compare the tick marks—not the label text—and record the revised original-image X coordinate for each longitude. Keep the calibration unchanged until the positions have been checked.

In [ ]:
# Step 2.70 — Show each top-edge longitude tick at native pixel resolution.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

# Earlier approximate X positions and corresponding printed longitudes.
tick_x_approx = [240, 490, 740, 990, 1250, 1500, 1770, 2020, 2270, 2520, 2775, 3025]
tick_labels = [
    "76°20'E", "76°25'E", "76°30'E", "76°35'E",
    "76°40'E", "76°45'E", "76°50'E", "76°55'E",
    "77°00'E", "77°05'E", "77°10'E", "77°15'E"
]

fig, axes = plt.subplots(4, 3, figsize=(15, 10))

for ax, x_approx, label in zip(axes.flat, tick_x_approx, tick_labels):
    crop = map_image.crop((x_approx - 35, 35, x_approx + 35, 115))
    ax.imshow(crop, interpolation="nearest")
    ax.set_title(f"{label} — approx. X={x_approx}px")
    ax.set_xticks(range(0, crop.width + 1, 10))
    ax.set_yticks(range(0, crop.height + 1, 20))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2.71 — Estimate Longitude Tick Centers

Estimate each top-edge longitude tick center from the original image by detecting the dark vertical tick stem below the frame line. The method reports the strongest candidate near each earlier approximate position and shows a diagnostic plot.

These are automated estimates, not confirmed measurements. Review the plot before using any revised positions in the calibration.

In [ ]:
# Step 2.71 — Estimate tick-stem centers from original image pixels.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

map_image = Image.open("PAMBA.jpg").convert("RGB")
image = np.asarray(map_image)

# Approximate X positions and longitude labels from the previous step.
tick_x_approx = np.array([
    240, 490, 740, 990, 1250, 1500,
    1770, 2020, 2270, 2520, 2775, 3025
])
tick_labels = [
    "76°20'E", "76°25'E", "76°30'E", "76°35'E",
    "76°40'E", "76°45'E", "76°50'E", "76°55'E",
    "77°00'E", "77°05'E", "77°10'E", "77°15'E"
]

# Search for dark pixels in the vertical tick-stem band.
# The horizontal frame line is above this band.
y_start, y_end = 78, 94
search_half_width = 35

gray = (
    0.299 * image[:, :, 0]
    + 0.587 * image[:, :, 1]
    + 0.114 * image[:, :, 2]
)

candidates = []

for label, x0 in zip(tick_labels, tick_x_approx):
    x_min = max(0, x0 - search_half_width)
    x_max = min(gray.shape[1], x0 + search_half_width + 1)

    band = gray[y_start:y_end, x_min:x_max]

    # Count dark pixels in each column; the tick stem should form a
    # concentrated vertical run beneath the frame.
    dark_pixel_counts = np.sum(band < 90, axis=0)

    best_local_x = int(np.argmax(dark_pixel_counts))
    estimated_x = x_min + best_local_x

    candidates.append({
        "Longitude_Label": label,
        "Earlier_Approx_X": int(x0),
        "Estimated_Tick_X": int(estimated_x),
        "Shift_Pixels": int(estimated_x - x0),
        "Dark_Pixel_Count": int(dark_pixel_counts[best_local_x])
    })

tick_candidates = pd.DataFrame(candidates)

print("AUTOMATED CANDIDATE POSITIONS — REVIEW BEFORE USING")
display(tick_candidates)

# Plot the original image band and the estimated tick centers.
fig, ax = plt.subplots(figsize=(18, 5))
ax.imshow(
    image[y_start:y_end, :, :],
    extent=(0, image.shape[1], y_end, y_start),
    aspect="auto",
    interpolation="nearest"
)

ax.scatter(
    tick_candidates["Estimated_Tick_X"],
    [86] * len(tick_candidates),
    color="red",
    marker="x",
    s=70,
    label="Estimated tick centers"
)

for _, row in tick_candidates.iterrows():
    ax.text(
        row["Estimated_Tick_X"],
        82,
        row["Longitude_Label"],
        rotation=60,
        fontsize=8,
        ha="left",
        va="bottom"
    )

ax.set_xlim(0, image.shape[1])
ax.set_ylim(y_end, y_start)
ax.set_xlabel("Original image X (pixels)")
ax.set_ylabel("Original image Y (pixels)")
ax.set_title("Step 2.71 — Candidate Tick Centers in the Original Image")
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## Step 2.72 — Isolate the Short Longitude Tick Marks

The previous detector selected long vertical grid lines because it searched too far below the frame. This step instead detects the horizontal frame line, then inspects only the narrow band immediately above it, where the short longitude tick marks appear.

The output is a candidate list and a close-up for visual verification. Do not use the candidate positions in the calibration until they are confirmed.

In [ ]:
# Step 2.72 — Detect short longitude tick marks above the top frame.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

map_image = Image.open("PAMBA.jpg").convert("RGB")
image = np.asarray(map_image)

gray = (
    0.299 * image[:, :, 0]
    + 0.587 * image[:, :, 1]
    + 0.114 * image[:, :, 2]
)

# The top frame is approximately around Y=93–95 px in the original image.
# Inspect only the band immediately above it, avoiding the long grid lines.
band_y_start, band_y_end = 78, 92
gray_band = gray[band_y_start:band_y_end, :]

# Count dark pixels per column in this narrow band.
dark_counts = np.sum(gray_band < 100, axis=0)

# Find candidate local peaks, then keep the strongest one near each
# previously identified longitude tick position.
tick_x_approx = np.array([
    240, 490, 740, 990, 1250, 1500,
    1770, 2020, 2270, 2520, 2775, 3025
])
tick_labels = [
    "76°20'E", "76°25'E", "76°30'E", "76°35'E",
    "76°40'E", "76°45'E", "76°50'E", "76°55'E",
    "77°00'E", "77°05'E", "77°10'E", "77°15'E"
]

search_half_width = 35
candidate_rows = []

for label, x0 in zip(tick_labels, tick_x_approx):
    x_min = max(0, x0 - search_half_width)
    x_max = min(gray.shape[1], x0 + search_half_width + 1)

    local_counts = dark_counts[x_min:x_max]
    estimated_x = x_min + int(np.argmax(local_counts))

    candidate_rows.append({
        "Longitude_Label": label,
        "Approx_X": int(x0),
        "Candidate_Tick_X": int(estimated_x),
        "Dark_Pixel_Count": int(dark_counts[estimated_x])
    })

tick_candidates = pd.DataFrame(candidate_rows)

print("CANDIDATE TICK POSITIONS — VISUAL REVIEW REQUIRED")
display(tick_candidates)

# Show the narrow detection band at native resolution.
fig, ax = plt.subplots(figsize=(18, 4))
ax.imshow(
    image[band_y_start:band_y_end, :, :],
    extent=(0, image.shape[1], band_y_end, band_y_start),
    aspect="auto",
    interpolation="nearest"
)

ax.scatter(
    tick_candidates["Candidate_Tick_X"],
    [85] * len(tick_candidates),
    color="red",
    marker="x",
    s=70,
    label="Candidate tick centers"
)

for _, row in tick_candidates.iterrows():
    ax.text(
        row["Candidate_Tick_X"],
        83,
        row["Longitude_Label"],
        rotation=60,
        fontsize=8,
        ha="left",
        va="bottom"
    )

ax.set_xlim(0, image.shape[1])
ax.set_ylim(band_y_end, band_y_start)
ax.set_xlabel("Original image X (pixels)")
ax.set_ylabel("Original image Y (pixels)")
ax.set_title("Step 2.72 — Narrow Band Above the Top Frame")
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## Step 2.73 — Verify Selected Longitude Reference Points

Inspect three well-spaced longitude ticks near the left, middle, and right of the top frame. For each, identify the center of the short vertical tick mark where it meets the frame line.

Record the original-image X coordinate for each tick. These checks will tell us whether the earlier approximate positions are consistently offset. Do not refit the calibration yet.

In [ ]:
# Step 2.73 — Inspect three longitude ticks at native resolution.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

# Three widely spaced reference ticks.
# These are approximate locations, not confirmed measurements.
reference_ticks = [
    ("76°25'E", 490),
    ("76°50'E", 1770),
    ("77°10'E", 2775),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (label, x0) in zip(axes, reference_ticks):
    # Crop around the tick stem and frame intersection.
    crop_box = (x0 - 25, 35, x0 + 26, 110)
    crop = map_image.crop(crop_box)

    ax.imshow(crop, interpolation="nearest")
    ax.set_title(f"{label}\nOriginal X ≈ {x0}px")
    ax.set_xlabel("Crop X (pixels)")
    ax.set_ylabel("Crop Y (pixels)")
    ax.set_xticks(range(0, crop.width + 1, 5))
    ax.set_yticks(range(0, crop.height + 1, 10))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2.74 — Inspect the Remaining Longitude Ticks

Use the same native-pixel crop method to inspect the remaining nine top-frame longitude ticks. For each crop, note the center of the short vertical tick stem where it meets the frame line.

The X positions shown are only starting estimates. Do not update the calibration yet.

In [ ]:
# Step 2.74 — Inspect remaining longitude ticks at native resolution.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

remaining_ticks = [
    ("76°20'E", 240),
    ("76°30'E", 740),
    ("76°35'E", 990),
    ("76°40'E", 1250),
    ("76°45'E", 1500),
    ("76°55'E", 2020),
    ("77°00'E", 2270),
    ("77°05'E", 2520),
    ("77°15'E", 3025),
]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))

for ax, (label, x0) in zip(axes.flat, remaining_ticks):
    crop_box = (x0 - 25, 35, x0 + 26, 110)
    crop = map_image.crop(crop_box)

    ax.imshow(crop, interpolation="nearest")
    ax.set_title(f"{label} — approximate original X={x0}px")
    ax.set_xlabel("Crop X (pixels)")
    ax.set_ylabel("Crop Y (pixels)")
    ax.set_xticks(range(0, crop.width + 1, 5))
    ax.set_yticks(range(0, crop.height + 1, 10))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2.75 — Inspect Longitude Tick Stems at Pixel Resolution

Display tight crops from the original `PAMBA.jpg` around the nine longitude ticks reviewed in Step 2.74. Each crop uses original-image pixel coordinates and a visible grid.

For each tick, identify the center of the short vertical stem where it meets the top frame line. Record the crop-X coordinate; the code converts it to the original-image X coordinate. These measurements remain provisional until reviewed.

In [ ]:
# Step 2.75 — Pixel-coordinate inspection of longitude tick stems.

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

map_image = Image.open("PAMBA.jpg").convert("RGB")

# Approximate positions from the previous steps.
ticks_to_check = [
    ("76°20'E", 240),
    ("76°30'E", 740),
    ("76°35'E", 990),
    ("76°40'E", 1250),
    ("76°45'E", 1500),
    ("76°55'E", 2020),
    ("77°00'E", 2270),
    ("77°05'E", 2520),
    ("77°15'E", 3025),
]

crop_half_width = 18
crop_top, crop_bottom = 38, 68
zoom_factor = 4

fig, axes = plt.subplots(3, 3, figsize=(15, 10))

for ax, (label, approx_x) in zip(axes.flat, ticks_to_check):
    x0 = max(0, approx_x - crop_half_width)
    x1 = min(map_image.width, approx_x + crop_half_width + 1)

    crop = map_image.crop((x0, crop_top, x1, crop_bottom))
    crop = crop.resize(
        (crop.width * zoom_factor, crop.height * zoom_factor),
        Image.Resampling.NEAREST
    )

    ax.imshow(crop, interpolation="nearest")
    ax.set_title(f"{label} — original X≈{approx_x}px")
    ax.set_xlabel("Crop X (pixels; 1 unit = 1 original pixel)")
    ax.set_ylabel("Original-image Y (pixels)")

    # Label the enlarged image in original-pixel coordinates.
    ax.set_xticks(
        np.arange(0, crop.width + 1, 5 * zoom_factor)
    )
    ax.set_xticklabels(
        np.arange(x0, x1 + 1, 5)
    )
    ax.set_yticks(
        np.arange(0, crop.height + 1, 5 * zoom_factor)
    )
    ax.set_yticklabels(
        np.arange(crop_top, crop_bottom + 1, 5)
    )
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2.76 — Capture the Full Longitude Tick Stems

The previous crops ended at original-image Y=68, before the tick stems reached the top frame line. Re-display the same ticks with a taller crop that includes both the label and the complete stem.

Use the stem’s center where it meets the frame line as the tick X position. Keep these measurements provisional until reviewed.

In [ ]:
# Step 2.76 — Show longitude labels and their full tick stems.

from PIL import Image
import matplotlib.pyplot as plt

map_image = Image.open("PAMBA.jpg").convert("RGB")

ticks_to_check = [
    ("76°20'E", 240),
    ("76°30'E", 740),
    ("76°35'E", 990),
    ("76°40'E", 1250),
    ("76°45'E", 1500),
    ("76°55'E", 2020),
    ("77°00'E", 2270),
    ("77°05'E", 2520),
    ("77°15'E", 3025),
]

half_width = 22
y_top, y_bottom = 35, 125
zoom = 3

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for ax, (label, approx_x) in zip(axes.flat, ticks_to_check):
    x0 = max(0, approx_x - half_width)
    x1 = min(map_image.width, approx_x + half_width + 1)

    crop = map_image.crop((x0, y_top, x1, y_bottom))
    crop = crop.resize(
        (crop.width * zoom, crop.height * zoom),
        Image.Resampling.NEAREST
    )

    ax.imshow(crop, interpolation="nearest")
    ax.set_title(f"{label} — original X≈{approx_x}px")
    ax.set_xlabel("Original-image X (pixels)")
    ax.set_ylabel("Original-image Y (pixels)")

    ax.set_xticks(range(0, crop.width + 1, 5 * zoom))
    ax.set_xticklabels(range(x0, x1 + 1, 5))
    ax.set_yticks(range(0, crop.height + 1, 10 * zoom))
    ax.set_yticklabels(range(y_top, y_bottom + 1, 10))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2.77 — Refit Longitude Calibration with Checked Ticks

Combine the previously recorded longitude tick positions with the newly inspected estimates. Fit a straight line from original-image X to longitude, then review the fit and residuals.

The newly estimated positions are still approximate. Treat this as a calibration check—not final georeferencing.

In [ ]:
# Step 2.77 — Refit longitude calibration using checked tick positions.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Longitude values in decimal degrees.
# Positions from the earlier reference table are retained;
# newly inspected positions are used for the nine reviewed ticks.
longitude_ticks = pd.DataFrame({
    "label": [
        "76°20'E", "76°25'E", "76°30'E", "76°35'E",
        "76°40'E", "76°45'E", "76°50'E", "76°55'E",
        "77°00'E", "77°05'E", "77°10'E", "77°15'E"
    ],
    "longitude": [
        76 + 20/60, 76 + 25/60, 76 + 30/60, 76 + 35/60,
        76 + 40/60, 76 + 45/60, 76 + 50/60, 76 + 55/60,
        77.0, 77 + 5/60, 77 + 10/60, 77 + 15/60
    ],
    "image_x": [
        241, 490, 744, 999, 1255, 1502,
        1770, 2020, 2270, 2520, 2775, 3025
    ]
})

X = longitude_ticks[["image_x"]].to_numpy()
y = longitude_ticks["longitude"].to_numpy()

model = LinearRegression()
model.fit(X, y)

longitude_ticks["predicted_longitude"] = model.predict(X)
longitude_ticks["residual_degrees"] = (
    longitude_ticks["longitude"] - longitude_ticks["predicted_longitude"]
)

# Approximate east-west distance at the map's mean latitude (~9.4°N).
metres_per_degree_lon = 111320 * np.cos(np.deg2rad(9.4))
longitude_ticks["residual_metres_approx"] = (
    longitude_ticks["residual_degrees"] * metres_per_degree_lon
)

rmse = np.sqrt(mean_squared_error(y, model.predict(X)))
r2 = r2_score(y, model.predict(X))

print(f"Longitude = {model.coef_[0]:.12f} × Image_X + {model.intercept_:.12f}")
print(f"R²: {r2:.8f}")
print(f"RMSE: {rmse:.8f} degrees")
print(
    "Maximum absolute residual: "
    f"{longitude_ticks['residual_degrees'].abs().max():.8f} degrees"
)
print(
    "Maximum absolute residual, approximate: "
    f"{longitude_ticks['residual_metres_approx'].abs().max():.1f} m"
)

display(longitude_ticks.round(6))

plt.figure(figsize=(10, 5))
plt.scatter(
    longitude_ticks["image_x"],
    longitude_ticks["longitude"],
    label="Tick references"
)
x_line = np.linspace(longitude_ticks["image_x"].min(),
                     longitude_ticks["image_x"].max(), 300)
plt.plot(x_line, model.predict(x_line.reshape(-1, 1)), label="Refitted line")
plt.xlabel("Original image X (pixels)")
plt.ylabel("Longitude (degrees east)")
plt.title("Step 2.77 — Refitted Longitude Calibration")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.axhline(0, color="black", linewidth=1)
plt.plot(
    longitude_ticks["image_x"],
    longitude_ticks["residual_metres_approx"],
    marker="o"
)
plt.xlabel("Original image X (pixels)")
plt.ylabel("Approx. east-west residual (m)")
plt.title("Step 2.77 — Longitude Calibration Residuals")
plt.grid(True, alpha=0.3)
plt.show()

## Step 2.78 — Measure Longitude Tick-Stem Centers

Estimate each tick’s X coordinate from the darkest vertical stem at the point where it joins the top frame line. This cell samples a narrow strip of the original image and plots the average darkness by X column to help locate each stem center.

Review the plotted peaks before accepting the coordinates. The image contains thick lines and labels, so automated peak detection is only a guide.

In [ ]:
# Step 2.78 — Inspect tick-stem centers using original image pixels.

from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

map_image = Image.open("PAMBA.jpg").convert("L")
image_array = np.asarray(map_image)

# Approximate centers used in Step 2.77.
ticks_to_check = [
    ("76°20'E", 241),
    ("76°25'E", 490),
    ("76°30'E", 744),
    ("76°35'E", 999),
    ("76°40'E", 1255),
    ("76°45'E", 1502),
    ("76°50'E", 1770),
    ("76°55'E", 2020),
    ("77°00'E", 2270),
    ("77°05'E", 2520),
    ("77°10'E", 2775),
    ("77°15'E", 3025),
]

# Narrow strip around the stem/frame junction; inspect before trusting.
x_half_width = 14
y_start, y_end = 82, 94

fig, axes = plt.subplots(4, 3, figsize=(15, 12))
measurements = []

for ax, (label, approx_x) in zip(axes.flat, ticks_to_check):
    x0 = max(0, approx_x - x_half_width)
    x1 = min(map_image.width, approx_x + x_half_width + 1)

    strip = image_array[y_start:y_end, x0:x1]

    # Dark pixels have low grayscale values.
    darkness_by_column = 255 - strip.mean(axis=0)
    local_x = np.arange(x0, x1)
    candidate_x = int(local_x[np.argmax(darkness_by_column)])

    measurements.append({
        "label": label,
        "previous_estimate_x": approx_x,
        "darkest_column_candidate_x": candidate_x,
    })

    ax.plot(local_x, darkness_by_column, marker=".", linewidth=1)
    ax.axvline(candidate_x, color="red", linestyle="--",
               label=f"candidate X={candidate_x}")
    ax.set_title(label)
    ax.set_xlabel("Original-image X (pixels)")
    ax.set_ylabel("Column darkness")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

display(pd.DataFrame(measurements))

## Step 2.78 — Refit Longitude Calibration Using Detected Tick Centers

Use the detected tick-center candidates as the pixel references and the printed longitude labels as the geographic references. Fit a linear model and inspect its residuals. These centers are still estimates, so this is a calibration check—not final georeferencing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Longitude labels in decimal degrees, converted from the printed DMS ticks
longitude_deg = np.array([
    76 + 20/60, 76 + 25/60, 76 + 30/60, 76 + 35/60,
    76 + 40/60, 76 + 45/60, 76 + 50/60, 76 + 55/60,
    77 + 0/60,  77 + 5/60,  77 + 10/60, 77 + 15/60
])

# Detected darkest-column candidates from Step 2.77
tick_x = np.array([242, 496, 749, 1002, 1255, 1508,
                   1761, 2015, 2268, 2521, 2774, 3027])

model = LinearRegression()
model.fit(tick_x.reshape(-1, 1), longitude_deg)

predicted_lon = model.predict(tick_x.reshape(-1, 1))
residual_deg = longitude_deg - predicted_lon

# Approximate east-west residual in metres at the map's latitude (~9.4°N)
residual_m = residual_deg * 111_320 * np.cos(np.deg2rad(9.4))

rmse_deg = np.sqrt(mean_squared_error(longitude_deg, predicted_lon))

print(f"Longitude = {model.coef_[0]:.12f} × Image_X + {model.intercept_:.12f}")
print(f"R²: {r2_score(longitude_deg, predicted_lon):.8f}")
print(f"RMSE: {rmse_deg:.8f} degrees")
print(f"Maximum absolute residual: {np.max(np.abs(residual_deg)):.8f} degrees")
print(f"Maximum absolute residual: {np.max(np.abs(residual_m)):.1f} m (approx.)")

residual_table = pd.DataFrame({
    "longitude_deg": longitude_deg,
    "detected_tick_x": tick_x,
    "predicted_longitude_deg": predicted_lon,
    "residual_deg": residual_deg,
    "residual_m_approx": residual_m
})
display(residual_table.round(6))

plt.figure(figsize=(10, 4))
plt.axhline(0, color="black", linewidth=1)
plt.plot(tick_x, residual_m, marker="o")
plt.xlabel("Original image X (pixels)")
plt.ylabel("East-west residual (m, approximate)")
plt.title("Step 2.78 — Longitude Calibration Residuals")
plt.grid(True, alpha=0.3)
plt.show()

## Step 2.79 — Calculate Provisional Map-Frame Coordinates

Apply the current longitude and latitude calibration models to the four corners of the map frame. This gives a provisional geographic extent for comparison with the printed graticule and independent basin references. It does not yet georeference or validate the basin boundary.

In [ ]:
import pandas as pd
# Approximate map-frame corners in original-image pixel coordinates
frame_corners = {
    "Top-left":     (107, 64),
    "Top-right":    (3221, 64),
    "Bottom-right": (3221, 2264),
    "Bottom-left":  (107, 2264),
}

# Latest Step 2.78 longitude fit
longitude_slope = model.coef_[0]
longitude_intercept = model.intercept_

# Existing provisional latitude fit from the earlier calibration
latitude_slope = -0.00034037
latitude_intercept = 9.817866

rows = []
for corner, (x, y) in frame_corners.items():
    longitude = longitude_slope * x + longitude_intercept
    latitude = latitude_slope * y + latitude_intercept

    rows.append({
        "corner": corner,
        "image_x": x,
        "image_y": y,
        "longitude": longitude,
        "latitude": latitude,
    })

frame_coordinates = pd.DataFrame(rows)
display(frame_coordinates.round(6))

print(
    "Provisional extent:",
    f"{frame_coordinates['longitude'].min():.6f} to "
    f"{frame_coordinates['longitude'].max():.6f}°E,",
    f"{frame_coordinates['latitude'].min():.6f} to "
    f"{frame_coordinates['latitude'].max():.6f}°N"
)

## Step 2.80 — Visualize the Provisionally Calibrated Map Frame

Crop the official Pamba basin map to its identified inner frame and display it using the provisional geographic extent calculated in Step 2.79. Inspect the resulting coordinate axes and map alignment before proceeding to georeferencing.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Open the original official map
image = Image.open("PAMBA.jpg").convert("RGB")

# Crop coordinates: left, top, right, bottom
frame_crop = image.crop((107, 64, 3221, 2264))

# Provisional geographic extent from Step 2.79
lon_min = 76.288748
lon_max = 77.313787
lat_min = 9.047268
lat_max = 9.796082

plt.figure(figsize=(12, 9))

plt.imshow(
    frame_crop,
    extent=[lon_min, lon_max, lat_min, lat_max],
    origin="upper",
    aspect="auto"
)

plt.xlabel("Longitude (°E)")
plt.ylabel("Latitude (°N)")
plt.title("Step 2.80 — Pamba Map Frame with Provisional Geographic Coordinates")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

print("Cropped map dimensions:", frame_crop.size)
print(f"Longitude extent: {lon_min:.6f}°E to {lon_max:.6f}°E")
print(f"Latitude extent: {lat_min:.6f}°N to {lat_max:.6f}°N")

## Step 2.81 — Validate Longitude and Latitude Calibration Residuals

Compare the current calibration models against the manually identified coordinate ticks on the original map. Calculate residuals in degrees and approximate metres for both axes. Use the results to assess whether the provisional map extent is reliable enough for the next georeferencing stage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Longitude reference ticks and updated detected centers
longitude_deg = np.array([
    76 + 20/60, 76 + 25/60, 76 + 30/60, 76 + 35/60,
    76 + 40/60, 76 + 45/60, 76 + 50/60, 76 + 55/60,
    77, 77 + 5/60, 77 + 10/60, 77 + 15/60
])

longitude_x = np.array([
    242, 496, 749, 1002, 1255, 1508,
    1761, 2015, 2268, 2521, 2774, 3027
])

# Latitude reference ticks and original-image Y positions
latitude_deg = np.array([
    9 + 5/60, 9 + 10/60, 9 + 15/60,
    9 + 20/60, 9 + 25/60, 9 + 30/60,
    9 + 35/60, 9 + 40/60, 9 + 45/60
])

latitude_y = np.array([
    2156.5, 1914, 1669, 1424, 1179,
    934, 689, 444, 199
])

# Current calibration equations
lon_predicted = model.predict(longitude_x.reshape(-1, 1))
lat_predicted = latitude_slope * latitude_y + latitude_intercept

# Residuals: reference minus predicted
lon_residual_deg = longitude_deg - lon_predicted
lat_residual_deg = latitude_deg - lat_predicted

# Approximate ground distances
lon_residual_m = (
    lon_residual_deg * 111320 * np.cos(np.deg2rad(9.4))
)
lat_residual_m = lat_residual_deg * 111320

# Display results
longitude_check = pd.DataFrame({
    "tick": longitude_deg,
    "image_x": longitude_x,
    "residual_m_approx": lon_residual_m
})

latitude_check = pd.DataFrame({
    "tick": latitude_deg,
    "image_y": latitude_y,
    "residual_m_approx": lat_residual_m
})

print("LONGITUDE CHECK")
display(longitude_check.round(3))

print("LATITUDE CHECK")
display(latitude_check.round(3))

print("SUMMARY")
print(f"Longitude MAE: {np.mean(np.abs(lon_residual_m)):.2f} m")
print(f"Longitude maximum error: {np.max(np.abs(lon_residual_m)):.2f} m")
print(f"Latitude MAE: {np.mean(np.abs(lat_residual_m)):.2f} m")
print(f"Latitude maximum error: {np.max(np.abs(lat_residual_m)):.2f} m")

# Residual plots
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

axes[0].axhline(0, color="black", linewidth=1)
axes[0].plot(longitude_x, lon_residual_m, marker="o")
axes[0].set_title("Longitude Calibration Residuals")
axes[0].set_xlabel("Original image X (pixels)")
axes[0].set_ylabel("East-west residual (m)")
axes[0].grid(alpha=0.3)

axes[1].axhline(0, color="black", linewidth=1)
axes[1].plot(latitude_y, lat_residual_m, marker="o")
axes[1].set_title("Latitude Calibration Residuals")
axes[1].set_xlabel("Original image Y (pixels)")
axes[1].set_ylabel("North-south residual (m)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 2.81 — Refit Longitude and Latitude Calibration

Fit independent linear transformations between original image pixels and geographic coordinates using the refined tick-centre positions.

Evaluate the new equations using residuals in metres, and compare their errors with the previous calibration. Retain the fitted equations only if the residual patterns and overall errors improve.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Refined longitude reference data
# --------------------------------------------------

longitude_deg = np.array([
    76 + 20/60, 76 + 25/60, 76 + 30/60, 76 + 35/60,
    76 + 40/60, 76 + 45/60, 76 + 50/60, 76 + 55/60,
    77, 77 + 5/60, 77 + 10/60, 77 + 15/60
])

longitude_x = np.array([
    242, 496, 749, 1002, 1255, 1508,
    1761, 2015, 2268, 2521, 2774, 3027
])

# --------------------------------------------------
# 2. Latitude reference data
# --------------------------------------------------

latitude_deg = np.array([
    9 + 5/60, 9 + 10/60, 9 + 15/60,
    9 + 20/60, 9 + 25/60, 9 + 30/60,
    9 + 35/60, 9 + 40/60, 9 + 45/60
])

latitude_y = np.array([
    2156.5, 1914, 1669, 1424, 1179,
    934, 689, 444, 199
])

# --------------------------------------------------
# 3. Fit new linear transformations
# --------------------------------------------------

# Longitude = slope * image_x + intercept
longitude_slope, longitude_intercept = np.polyfit(
    longitude_x, longitude_deg, 1
)

# Latitude = slope * image_y + intercept
latitude_slope_new, latitude_intercept_new = np.polyfit(
    latitude_y, latitude_deg, 1
)

# Predicted coordinates at reference tick positions
lon_predicted_new = (
    longitude_slope * longitude_x + longitude_intercept
)

lat_predicted_new = (
    latitude_slope_new * latitude_y + latitude_intercept_new
)

# --------------------------------------------------
# 4. Calculate residuals
# --------------------------------------------------

lon_residual_deg = longitude_deg - lon_predicted_new
lat_residual_deg = latitude_deg - lat_predicted_new

# Approximate conversion to metres
lon_residual_m = (
    lon_residual_deg * 111320 * np.cos(np.deg2rad(9.4))
)

lat_residual_m = lat_residual_deg * 111320

# --------------------------------------------------
# 5. Calculate calibration metrics
# --------------------------------------------------

def calculate_metrics(reference, predicted, residual_m):
    return {
        "R2": 1 - np.sum((reference - predicted) ** 2) /
              np.sum((reference - np.mean(reference)) ** 2),
        "MAE_m": np.mean(np.abs(residual_m)),
        "RMSE_m": np.sqrt(np.mean(residual_m ** 2)),
        "Max_Error_m": np.max(np.abs(residual_m))
    }

longitude_metrics = calculate_metrics(
    longitude_deg, lon_predicted_new, lon_residual_m
)

latitude_metrics = calculate_metrics(
    latitude_deg, lat_predicted_new, lat_residual_m
)

# --------------------------------------------------
# 6. Display fitted equations
# --------------------------------------------------

print("UPDATED LONGITUDE EQUATION")
print(
    f"Longitude = {longitude_slope:.12f} * Image_X "
    f"+ {longitude_intercept:.12f}"
)

print("\nUPDATED LATITUDE EQUATION")
print(
    f"Latitude = {latitude_slope_new:.12f} * Image_Y "
    f"+ {latitude_intercept_new:.12f}"
)

# --------------------------------------------------
# 7. Display residual tables
# --------------------------------------------------

longitude_check_new = pd.DataFrame({
    "tick": longitude_deg,
    "image_x": longitude_x,
    "predicted_longitude": lon_predicted_new,
    "residual_m": lon_residual_m
})

latitude_check_new = pd.DataFrame({
    "tick": latitude_deg,
    "image_y": latitude_y,
    "predicted_latitude": lat_predicted_new,
    "residual_m": lat_residual_m
})

print("\nLONGITUDE CALIBRATION CHECK")
display(longitude_check_new.round(6))

print("\nLATITUDE CALIBRATION CHECK")
display(latitude_check_new.round(6))

print("\nCALIBRATION METRICS")
display(pd.DataFrame(
    [longitude_metrics, latitude_metrics],
    index=["Longitude", "Latitude"]
).round(6))

# --------------------------------------------------
# 8. Plot updated residuals
# --------------------------------------------------

fig, axes = plt.subplots(2, 1, figsize=(11, 7))

axes[0].axhline(0, color="black", linewidth=1)
axes[0].plot(
    longitude_x, lon_residual_m,
    marker="o", label="Longitude residual"
)
axes[0].set_title("Step 2.82 — Updated Longitude Residuals")
axes[0].set_xlabel("Original image X (pixels)")
axes[0].set_ylabel("East-west residual (m)")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].axhline(0, color="black", linewidth=1)
axes[1].plot(
    latitude_y, lat_residual_m,
    marker="o", label="Latitude residual"
)
axes[1].set_title("Step 2.82 — Updated Latitude Residuals")
axes[1].set_xlabel("Original image Y (pixels)")
axes[1].set_ylabel("North-south residual (m)")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()